<a href="https://colab.research.google.com/github/akhi-lbj/SQLGuard/blob/main/SQLGuard_Full_DEV_BIRRD_With1st_Ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Create isolated full_dev directory inside bird_data
FULL_DEV_DIR = "/content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev"
os.makedirs(FULL_DEV_DIR, exist_ok=True)
os.chdir(FULL_DEV_DIR)

print(f"Target directory set to: {os.getcwd()}")

In [ ]:
%%bash
# Download official BIRD Dev package directly into full_dev/
wget -q --show-progress -O dev.zip "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip"

# Unzip and clean up
unzip -q dev.zip
rm dev.zip

echo "Full BIRD Dev dataset extraction complete inside full_dev/!"

Full BIRD Dev dataset extraction complete inside full_dev/!



     0K .......... .......... .......... .......... ..........  0% 1.47M 3m44s
    50K .......... .......... .......... .......... ..........  0%  235K 13m50s
   100K .......... .......... .......... .......... ..........  0% 1.42M 10m31s
   150K .......... .......... .......... .......... ..........  0%  255K 13m24s
   200K .......... .......... .......... .......... ..........  0% 1.40M 11m31s
   250K .......... .......... .......... .......... ..........  0% 1.29M 10m18s
   300K .......... .......... .......... .......... ..........  0% 1.36M 9m24s
   350K .......... .......... .......... .......... ..........  0% 1.26M 8m46s
   400K .......... .......... .......... .......... ..........  0% 1.39M 8m14s
   450K .......... .......... .......... .......... ..........  0%  727K 8m11s
   500K .......... .......... .......... .......... ..........  0% 1.54M 7m46s
   550K .......... .......... .......... .......... ..........  0% 1.43M 7m26s
   600K .......... .......... .......... .....

In [ ]:
ls

dev_20240627/


In [ ]:
cd dev_20240627/

/content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627


In [ ]:
ls

ablation_results_separate/  dev.sql               results/
dev_databases/              dev_tables.json       results_7b/
dev.json                    dev_tied_append.json  results_7b_with_evidence/


In [ ]:
!unzip -q dev_databases.zip

In [ ]:
!ls -la
!find dev_databases/ -name "*.sqlite" | wc -l

total 1186
drwx------ 2 root root   4096 Aug 28 10:20 dev_databases
-rw------- 1 root root 741332 Jun 27  2024 dev.json
-rw------- 1 root root 272006 Jun 27  2024 dev.sql
-rw------- 1 root root 158350 Sep 25  2023 dev_tables.json
-rw------- 1 root root  25509 Sep 19  2023 dev_tied_append.json
drwx------ 2 root root   4096 Aug 28 11:38 results
drwx------ 2 root root   4096 Aug 28 16:07 results_7b
drwx------ 2 root root   4096 Aug 28 18:42 results_7b_with_evidence
11


In [ ]:
!rm -rf __MACOSX dev_databases.zip

In [ ]:
import json
import sqlite3
from pathlib import Path

# Load dev questions
with open("dev.json", "r") as f:
    dev_data = json.load(f)

print(f"Total questions: {len(dev_data)}")

# Test first sample against its database
sample = dev_data[0]
db_id = sample["db_id"]
db_path = Path(f"dev_databases/{db_id}/{db_id}.sqlite")

print(f"\nSample DB: {db_id}")
print(f"Question: {sample['question']}")
print(f"Evidence: {sample.get('evidence', 'None')}")
print(f"Gold SQL: {sample['SQL']}")

if db_path.exists():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute(sample["SQL"])
    print(f"Query Result: {cursor.fetchall()[:3]}")
    conn.close()
    print("\nDatabase connection and SQL execution successful.")
else:
    print(f"\nError: Database file not found at {db_path}")

Total questions: 1534

Sample DB: california_schools
Question: What is the highest eligible free rate for K-12 students in the schools in Alameda County?
Evidence: Eligible free rate for K-12 = `Free Meal Count (K-12)` / `Enrollment (K-12)`
Gold SQL: SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1
Query Result: [(1.0,)]

Database connection and SQL execution successful.


In [ ]:
!pip install -q huggingface_hub transformers accelerate

In [ ]:
# Download BIRD fine-tuned checkpoint
!python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='seeklhy/codes-3b-bird', local_dir='./codes-3b-bird')"

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/2.18G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/12.2G [00:00<?, ?B/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/12.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 564/12.2G [00:00<3816:14:01, 886B/s]
Reconstructing (incomplete total...):   0% 564/12.2G [00:00<3816:14:01, 886B/s]
Reconstructing (incomplete total...):   0% 2.19k/12.2G [00:00<3816:14:36, 886B/s]
Reconstructing (incomplete total...):   0% 34.9k/12.2G [00:00<3816:14:01, 886B/s]

Fetching 11 files:   9% 1/11 [00:00<00:07,  1.39it/s]
Reconstructing (incomplete total...):   0% 35.9k/12.2G [00:00<3816:22:18, 886B/s]

Fetching 11 files:  45% 5/11 [00:00<00:00,  7.75it/s]
R

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "seeklhy/codes-3b-bird"

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully onto GPU.")

Loading seeklhy/codes-3b-bird...


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/34.4k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully onto GPU.


In [ ]:
prompt = """-- Database Schema:
-- Table frpm: [CDSCode, County Name, Free Meal Count (K-12), Enrollment (K-12)]
-- Table schools: [CDSCode, School, City]
-- Question: What is the highest eligible free rate for K-12 students in the schools in Alameda County?
-- Evidence: Eligible free rate for K-12 = Free Meal Count (K-12) / Enrollment (K-12)
SELECT"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False,  # Greedy decoding / beam search
        num_beams=4
    )

generated_sql = "SELECT" + tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\nGenerated SQL:")
print(generated_sql)


Generated SQL:
SELECT max(Free Meal Count (K-12) / Enrollment (K-12)) FROM frpm WHERE County Name = 'Alameda County'


In [ ]:
ls

codes-3b-bird/  dev.json  dev_tables.json
dev_databases/  dev.sql   dev_tied_append.json


In [ ]:
import os
import re
import json
import time
import shutil
import sqlite3
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Dict, Any, TypedDict
from pydantic import BaseModel, Field
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sqlglot
from sqlglot import parse_one, exp
from openai import OpenAI
from tqdm import tqdm
from langgraph.graph import StateGraph, START, END

# =====================================================================
# 0. DIRECTORY PATHS, FULL_DEV RESOLUTION & RESULTS CLEANING
# =====================================================================
DRIVE_BASE_DIR = "/content/drive/MyDrive/SQLGuard_BIRD"
DRIVE_FULL_DEV_CANDIDATES = [
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev", "dev_20240627"),
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev"),
    os.path.join(DRIVE_BASE_DIR, "bird_data")
]

LOCAL_WORKING_DIR = "/content/sqlguard_run"
LOCAL_FULL_DEV_DIR = os.path.join(LOCAL_WORKING_DIR, "full_dev")
LOCAL_RESULTS_DIR = os.path.join(LOCAL_FULL_DEV_DIR, "results")
LOCAL_RESULTS_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_results.jsonl")
LOCAL_MEMORY_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_failure_memory.json")

os.makedirs(LOCAL_WORKING_DIR, exist_ok=True)
file_lock = threading.Lock()
cache_lock = threading.Lock()
gpu_model_lock = threading.Lock()


def resolve_drive_source_dir() -> str:
    """Locates the directory containing dev.json and dev_databases."""
    for candidate in DRIVE_FULL_DEV_CANDIDATES:
        if os.path.exists(candidate):
            has_json = any(os.path.exists(os.path.join(candidate, f)) for f in ["dev.json", "dev_20240627/dev.json"])
            if has_json:
                return candidate
    return DRIVE_FULL_DEV_CANDIDATES[1]


def get_drive_results_dir() -> str:
    """Returns the target Drive results directory inside full_dev."""
    source_dir = resolve_drive_source_dir()
    return os.path.join(source_dir, "results")


def clean_and_setup_results_dir():
    """Wipes any previous execution artifacts from full_dev/results both locally and on Drive."""
    if os.path.exists(LOCAL_RESULTS_DIR):
        print(f">>> Removing previous local results in {LOCAL_RESULTS_DIR}...")
        shutil.rmtree(LOCAL_RESULTS_DIR)
    os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

    drive_results_dir = get_drive_results_dir()
    if os.path.exists(drive_results_dir):
        print(f">>> Removing previous Drive results in {drive_results_dir}...")
        shutil.rmtree(drive_results_dir)
    os.makedirs(drive_results_dir, exist_ok=True)

    print(">>> [READY] Fresh full_dev/results directory initialized.")


def setup_local_colab_environment():
    """Copies dataset files from Google Drive to local NVMe storage."""
    source_dir = resolve_drive_source_dir()
    if os.path.exists(source_dir) and not os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Staging full_dev dataset from {source_dir} to local NVMe ({LOCAL_FULL_DEV_DIR})...")
        shutil.copytree(source_dir, LOCAL_FULL_DEV_DIR)
        print(">>> Staging complete!")
    elif os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Local full_dev dataset ready at {LOCAL_FULL_DEV_DIR}.")


def sync_results_to_drive():
    """Atomically syncs incremental results and failure memory back to full_dev/results on Drive."""
    try:
        drive_results_dir = get_drive_results_dir()
        os.makedirs(drive_results_dir, exist_ok=True)
        if os.path.exists(LOCAL_RESULTS_FILE):
            shutil.copy(LOCAL_RESULTS_FILE, os.path.join(drive_results_dir, "sqlguard_results.jsonl"))
        if os.path.exists(LOCAL_MEMORY_FILE):
            shutil.copy(LOCAL_MEMORY_FILE, os.path.join(drive_results_dir, "sqlguard_failure_memory.json"))
        print("\n>>> [SYNC SUCCESS] Checkpointed results to Drive: full_dev/results/")
    except Exception as e:
        print(f"\n>>> [SYNC ERROR] Drive backup failed: {e}")


# =====================================================================
# 1. K2-THINK-V2 ROTATING API MANAGER
# =====================================================================
K2_BASE_URL = os.getenv("K2_BASE_URL", "https://api.k2think.ai/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "MBZUAI-IFM/K2-Think-v2")

API_KEYS = [
    "IFM-93mLmFQS2bfYZZIY",
    "IFM-SQJYZjtO76Kk86de",
    "IFM-BDZ81VHhqLyYn6Ka"
]


def clean_reasoning_output(raw_text: Optional[str]) -> str:
    """Strips <think> tags, markdown fences, and extracts clean JSON/SQL."""
    if not raw_text:
        return ""
    cleaned = re.sub(r"<think>.*?</think>", "", str(raw_text), flags=re.DOTALL).strip()

    if "```json" in cleaned:
        cleaned = cleaned.split("```json")[1].split("```")[0].strip()
    elif "```sql" in cleaned:
        cleaned = cleaned.split("```sql")[1].split("```")[0].strip()
    elif "```" in cleaned:
        cleaned = cleaned.split("```")[1].split("```")[0].strip()

    if not cleaned.startswith("{") and "select" in cleaned.lower():
        select_pos = cleaned.lower().find("select")
        if select_pos != -1:
            cleaned = cleaned[select_pos:].strip()
            cleaned = cleaned.replace("```", "").strip()

    return cleaned


class K2DynamicKeyManager:
    """Round-robin load balancer across active API keys with latency fallback."""
    def __init__(self, keys: List[str], base_url: str):
        self.keys = [k for k in keys if k and not k.startswith("YOUR_")]
        self.base_url = base_url
        self.clients = {k: OpenAI(base_url=self.base_url, api_key=k, timeout=60.0) for k in self.keys}
        self.index = 0
        self.lock = threading.Lock()

    def get_client_and_key(self) -> tuple[OpenAI, str]:
        with self.lock:
            key = self.keys[self.index % len(self.keys)]
            self.index += 1
            return self.clients[key], key

    def rotate_away_from(self, slow_key: str):
        with self.lock:
            if self.keys[self.index % len(self.keys)] == slow_key:
                self.index += 1

    def execute_chat_completion(self, prompt: str, max_retries: int = 2) -> str:
        for attempt in range(max_retries + 1):
            client, active_key = self.get_client_and_key()
            start_time = time.time()
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0
                )
                if time.time() - start_time > 50.0:
                    self.rotate_away_from(active_key)
                content = resp.choices[0].message.content if resp.choices else ""
                return clean_reasoning_output(content)
            except Exception:
                self.rotate_away_from(active_key)
                if attempt == max_retries:
                    return ""
                time.sleep(1.0)
        return ""


llm_manager = K2DynamicKeyManager(API_KEYS, K2_BASE_URL)


# =====================================================================
# 2. LOCAL CODES-3B ENGINE (GPU INFERENCE)
# =====================================================================
class LocalCodeSEngine:
    """Thread-safe GPU inference manager for CodeS-3B."""
    def __init__(self, model_id: str = "seeklhy/codes-3b-bird"):
        print(f">>> Loading local SQL generator: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        print(">>> CodeS-3B loaded successfully onto GPU.")

    def generate_sql(self, schema_str: str, question: str, evidence: str, contract_guidance: str) -> str:
        prompt = (
            f"-- Database Schema:\n{schema_str}\n"
            f"-- Question: {question}\n"
            f"-- Evidence: {evidence}\n"
            f"-- Semantic Contract Guidance: {contract_guidance}\n"
            f"SELECT"
        )
        with gpu_model_lock:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=160,
                    pad_token_id=self.tokenizer.eos_token_id,
                    do_sample=False,
                    num_beams=4
                )
            generated = "SELECT" + self.tokenizer.decode(
                output_ids[0][inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )
        return clean_reasoning_output(generated)


codes_engine = LocalCodeSEngine()


# =====================================================================
# 3. THREAD-SAFE PERSISTENT FAILURE MEMORY
# =====================================================================
class PersistentFailureMemory:
    """Maintains an on-disk few-shot repository of repaired SQL patterns inside full_dev/results/."""
    def __init__(self, memory_filepath: str = LOCAL_MEMORY_FILE):
        self.filepath = memory_filepath
        self.memory: List[Dict[str, Any]] = []
        self.reload()

    def reload(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r") as f:
                    self.memory = json.load(f)
            except Exception:
                self.memory = []
        else:
            self.memory = []

    def record_repair(self, question: str, failed_sql: str, error_feedback: str, fixed_sql: str, error_types: List[str]):
        entry = {
            "question": question,
            "failed_sql": failed_sql,
            "error_feedback": error_feedback,
            "fixed_sql": fixed_sql,
            "error_types": error_types
        }
        with file_lock:
            self.memory.append(entry)
            with open(self.filepath, "w") as f:
                json.dump(self.memory, f, indent=2)

    def retrieve_similar_repairs(self, current_errors: List[str], max_examples: int = 2) -> str:
        with file_lock:
            if not self.memory:
                return ""
            mem_snapshot = list(self.memory)

        retrieved = []
        for entry in reversed(mem_snapshot):
            for err in current_errors:
                if any(err_type.lower() in err.lower() for err_type in entry.get("error_types", [])):
                    retrieved.append(entry)
                    break
            if len(retrieved) >= max_examples:
                break

        if not retrieved:
            retrieved = mem_snapshot[-max_examples:]

        formatted_cases = []
        for idx, item in enumerate(retrieved, 1):
            formatted_cases.append(
                f"[Past Repair Example #{idx}]\n"
                f"Question: {item['question']}\n"
                f"Failed Query: {item['failed_sql']}\n"
                f"Errors: {item['error_feedback']}\n"
                f"Corrected SQL: {item['fixed_sql']}"
            )
        return "\n\n".join(formatted_cases)


global_memory = PersistentFailureMemory()


# =====================================================================
# 4. 7-DIMENSIONAL SEMANTIC CONTRACT SCHEMA (Γ)
# =====================================================================
class TargetProjection(BaseModel):
    entity: str = Field(description="Summary of target projection")
    output_columns: List[str] = Field(default_factory=list, description="Projected expressions")
    granularity: Optional[str] = Field(default=None, description="Granularity level")

class SchemaLinks(BaseModel):
    required_tables: List[str] = Field(default_factory=list, description="Required tables")
    required_columns: List[str] = Field(default_factory=list, description="Required columns")
    join_keys: List[str] = Field(default_factory=list, description="Join paths")

class Analytics(BaseModel):
    aggregations: List[str] = Field(default_factory=list, description="COUNT, AVG, SUM, MIN, MAX")
    group_by: List[str] = Field(default_factory=list, description="Grouping columns or date slice expressions")
    having: List[str] = Field(default_factory=list, description="HAVING conditions")

class RankingCardinality(BaseModel):
    order_by: List[str] = Field(default_factory=list, description="Sort expressions")
    direction: Optional[str] = Field(default=None, description="ASC or DESC")
    limit: Optional[int] = Field(default=None, description="LIMIT top-k cap")

class SemanticContract(BaseModel):
    target_projection: TargetProjection
    schema_links: SchemaLinks
    predicates: List[str] = Field(default_factory=list, description="WHERE filters preserving INTEGER affinity")
    analytics: Analytics
    ranking_cardinality: RankingCardinality
    read_only: bool = Field(default=True, description="Strict read-only safety flag")
    ambiguity_flag: bool = Field(default=False, description="Ambiguity status")


# =====================================================================
# 5. COMPACT SCHEMA EXTRACTOR & CACHING
# =====================================================================
SCHEMA_CACHE: Dict[str, str] = {}


def extract_compact_schema(db_path: Optional[str]) -> str:
    """Builds a token-efficient, type-annotated SQLite schema description."""
    if not db_path or not os.path.exists(db_path):
        return "Schema unavailable."

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%';")
        tables = [r[0] for r in cursor.fetchall()]

        schema_lines = []
        for table_name in tables:
            cursor.execute(f"PRAGMA table_info('{table_name}');")
            cols = cursor.fetchall()
            col_desc = [f"{c[1]} ({c[2].upper() or 'TEXT'})" for c in cols]
            schema_lines.append(f"TABLE {table_name} (\n  " + ", ".join(col_desc) + "\n)")

            cursor.execute(f"PRAGMA foreign_key_list('{table_name}');")
            for fk in cursor.fetchall():
                schema_lines.append(f"-- FK: {table_name}.{fk[3]} -> {fk[2]}.{fk[4]}")

            samples = []
            sampled_count = 0
            for col in cols:
                if sampled_count >= 4:
                    break
                col_name = col[1]
                cursor.execute(f"SELECT DISTINCT \"{col_name}\" FROM \"{table_name}\" WHERE \"{col_name}\" IS NOT NULL LIMIT 3;")
                vals = [r[0] for r in cursor.fetchall() if r[0] is not None]
                if vals:
                    samples.append(f"{col_name}: {vals}")
                    sampled_count += 1
            if samples:
                schema_lines.append(f"-- [{table_name} Samples]: " + " | ".join(samples))

        conn.close()
        return "\n".join(schema_lines)
    except Exception as e:
        return f"Error reading schema: {e}"


def get_cached_schema(db_id: str, db_path: Optional[str]) -> str:
    with cache_lock:
        if db_id in SCHEMA_CACHE:
            return SCHEMA_CACHE[db_id]

    compact_schema = extract_compact_schema(db_path)
    with cache_lock:
        SCHEMA_CACHE[db_id] = compact_schema
    return compact_schema


# =====================================================================
# 6. NORMALIZED HYBRID AST VALIDATOR (sqlglot)
# =====================================================================
class SQLGuardValidator:
    def validate(self, sql: str, contract: SemanticContract) -> Dict[str, Any]:
        errors = []
        error_types = []

        if not sql or not sql.strip():
            return {"passed": False, "errors": ["Generated SQL was empty."], "error_types": ["Syntax"]}

        try:
            parsed = parse_one(sql, read="sqlite")
        except Exception as e:
            return {"passed": False, "errors": [f"AST Parse Error: {str(e)}"], "error_types": ["Syntax"]}

        if not isinstance(parsed, exp.Select):
            return {
                "passed": False,
                "errors": ["Unit Test [V_safety] Failed: Non-SELECT operation blocked."],
                "error_types": ["Safety"]
            }

        query_tables = {t.name.lower().strip("`'\"[] ") for t in parsed.find_all(exp.Table)}
        query_columns_bare = {c.name.lower().strip("`'\"[] ") for c in parsed.find_all(exp.Column)}

        # 1. Table Verification
        for req_t in contract.schema_links.required_tables:
            req_t_clean = req_t.lower().strip("`'\"[] ")
            if req_t_clean and req_t_clean not in query_tables:
                errors.append(f"Unit Test [Schema Table] Failed: Required table '{req_t}' missing.")
                error_types.append("SchemaTable")

        # 2. Column Verification
        for req_c in contract.schema_links.required_columns:
            req_clean = req_c.lower().strip("`'\"[] ")
            req_bare = req_clean.split(".")[-1].strip("`'\"[] ")
            if req_bare and req_bare not in query_columns_bare:
                errors.append(f"Unit Test [Schema Column] Failed: Required column '{req_c}' missing.")
                error_types.append("SchemaColumn")

        # 3. Aggregations
        if contract.analytics.aggregations:
            ast_funcs = set()
            if parsed.find(exp.Count): ast_funcs.add("count")
            if parsed.find(exp.Sum): ast_funcs.add("sum")
            if parsed.find(exp.Avg): ast_funcs.add("avg")
            if parsed.find(exp.Max): ast_funcs.add("max")
            if parsed.find(exp.Min): ast_funcs.add("min")

            for req_agg in contract.analytics.aggregations:
                req_clean = req_agg.lower().strip()
                matched = any(kw in req_clean and kw in ast_funcs for kw in ["count", "sum", "avg", "max", "min"])
                if not matched:
                    errors.append(f"Unit Test [Aggregation] Failed: Missing required function '{req_agg}'.")
                    error_types.append("Aggregation")

        # 4. Predicates
        if contract.predicates:
            has_where = parsed.find(exp.Where) is not None
            has_having = parsed.find(exp.Having) is not None
            if not (has_where or has_having):
                errors.append("Unit Test [Predicate] Failed: Filters specified in contract but WHERE/HAVING missing.")
                error_types.append("Predicate")

        # 5. Group By
        if contract.analytics.group_by and not parsed.find(exp.Group):
            errors.append("Unit Test [GroupBy] Failed: Contract specifies grouping but GROUP BY clause missing.")
            error_types.append("GroupBy")

        # 6. Order By & Limit
        if contract.ranking_cardinality.direction and not parsed.find(exp.Order):
            errors.append("Unit Test [Ranking] Failed: Contract specifies sort order but ORDER BY clause missing.")
            error_types.append("Ranking")

        if contract.ranking_cardinality.limit is not None and not parsed.find(exp.Limit):
            errors.append("Unit Test [Limit] Failed: Contract specifies top-k cap but LIMIT clause missing.")
            error_types.append("Limit")

        return {
            "passed": len(errors) == 0,
            "errors": errors,
            "error_types": list(set(error_types))
        }


# =====================================================================
# 7. LANGGRAPH WORKFLOW NODES
# =====================================================================
class SQLGuardState(TypedDict):
    question: str
    evidence: str
    db_id: str
    db_path: str
    gold_sql: str
    schema_metadata: str
    contract: Optional[SemanticContract]
    current_sql: str
    validation_passed: bool
    validation_errors: List[str]
    validation_error_types: List[str]
    attempt_count: int
    max_attempts: int
    initial_failed_sql: str
    initial_errors: List[str]
    initial_error_types: List[str]
    ex_passed: bool
    execution_result: Optional[List[Any]]
    audit_record: Dict[str, Any]


def schema_linker_node(state: SQLGuardState) -> Dict[str, Any]:
    schema_meta = get_cached_schema(state["db_id"], state["db_path"])
    return {"schema_metadata": schema_meta}


def intent_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    prompt = f"""You are the Intent Agent for SQLGuard. Extract a 7-dimensional Semantic Contract as a JSON object.

### MANDATORY INSTRUCTIONS:
1. DOMAIN EVIDENCE GROUNDING: Strictly follow domain evidence. If evidence indicates date slicing (e.g. SUBSTR(Date, 5, 2) for month, SUBSTR(Date, 1, 4) for year), use that exact expression in output_columns and group_by.
2. SQLITE TYPE COMPLIANCE: If column type is INTEGER (e.g. Date 201301), do NOT wrap numeric numbers in single quotes (use `Date BETWEEN 201301 AND 201312`).
3. PROJECTION PRECISION: Output ONLY the requested attribute or expression in output_columns.

### BENCHMARK EVALUATION GROUNDING RULES:
1. NAME PROJECTION: When asked for a person's name or full name, project two separate columns `first_name, last_name` (or `forename, surname`). DO NOT concatenate with `|| ' ' ||`.
2. CASE-INSENSITIVE TEXT FILTERS: For string equality checks in WHERE clauses, use `COLLATE NOCASE` or `LIKE` (e.g. `Segment = 'Discount' COLLATE NOCASE`).
3. PROJECTION MINIMALISM: Project ONLY the exact attribute requested. Do not include extra tie-breaker columns or IDs in SELECT unless explicitly requested.
4. NULL-SAFE SUMS: Always provide `ELSE 0` in conditional aggregation (e.g., `SUM(CASE WHEN condition THEN val ELSE 0 END)`).

Schema:
{state["schema_metadata"]}

User Question: {state["question"]}
Domain Evidence: {state["evidence"]}

JSON Schema:
{json.dumps(SemanticContract.model_json_schema())}

Output ONLY the raw JSON object."""

    raw_json = llm_manager.execute_chat_completion(prompt)
    try:
        contract = SemanticContract.model_validate_json(raw_json)
    except Exception:
        contract = SemanticContract(
            target_projection=TargetProjection(entity=state["question"]),
            schema_links=SchemaLinks(),
            analytics=Analytics(),
            ranking_cardinality=RankingCardinality()
        )
    return {"contract": contract}


def generator_decomposer_node(state: SQLGuardState) -> Dict[str, Any]:
    contract_str = ""
    if state["contract"]:
        contract_dict = state["contract"].model_dump(exclude_none=True)
        contract_str = json.dumps(contract_dict)

    generated_sql = codes_engine.generate_sql(
        schema_str=state["schema_metadata"],
        question=state["question"],
        evidence=state["evidence"],
        contract_guidance=contract_str
    )
    return {"current_sql": generated_sql}


def hybrid_validator_node(state: SQLGuardState) -> Dict[str, Any]:
    validator = SQLGuardValidator()
    val = validator.validate(state["current_sql"], state["contract"])

    updates = {
        "validation_passed": val["passed"],
        "validation_errors": val["errors"],
        "validation_error_types": val.get("error_types", [])
    }

    if not val["passed"] and state["attempt_count"] == 0:
        updates["initial_failed_sql"] = state["current_sql"]
        updates["initial_errors"] = val["errors"]
        updates["initial_error_types"] = val.get("error_types", [])

    return updates


def repair_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    memory_ctx = global_memory.retrieve_similar_repairs(state["validation_errors"])
    contract_json = state["contract"].model_dump_json() if state["contract"] else "{}"

    prompt = f"""Repair the following SQLite query to satisfy the Semantic Contract and Domain Evidence.

Schema:
{state["schema_metadata"]}

Question: {state["question"]}
Evidence: {state["evidence"]}
Semantic Contract: {contract_json}

[FAILED CANDIDATE SQL]:
{state["current_sql"]}

[CONTRACT VALIDATION ERRORS]:
{chr(10).join(state["validation_errors"])}
"""
    if memory_ctx:
        prompt += f"\n[SIMILAR PAST SUCCESSFUL REPAIRS]:\n{memory_ctx}\n"

    prompt += "\nOutput raw repaired SQL inside a ```sql codeblock."

    repaired_sql = llm_manager.execute_chat_completion(prompt)
    if not repaired_sql:
        repaired_sql = state["current_sql"]

    return {
        "current_sql": repaired_sql,
        "attempt_count": state["attempt_count"] + 1
    }


def execution_gate_node(state: SQLGuardState) -> Dict[str, Any]:
    ex_passed = False
    pred_res = None

    if state["validation_passed"] and state["db_path"] and os.path.exists(state["db_path"]):
        conn = sqlite3.connect(state["db_path"])
        cursor = conn.cursor()
        try:
            cursor.execute(state["current_sql"])
            pred_res = cursor.fetchall()

            cursor.execute(state["gold_sql"])
            gold_res = cursor.fetchall()

            ex_passed = (pred_res == gold_res or set(pred_res) == set(gold_res))
        except Exception:
            ex_passed = False
        finally:
            conn.close()

    if state["validation_passed"] and state["attempt_count"] > 0 and state["initial_failed_sql"]:
        global_memory.record_repair(
            question=state["question"],
            failed_sql=state["initial_failed_sql"],
            error_feedback="\n".join(state["initial_errors"]),
            fixed_sql=state["current_sql"],
            error_types=state["initial_error_types"]
        )

    audit_record = {
        "question": state["question"],
        "db_id": state["db_id"],
        "contract": state["contract"].model_dump() if state["contract"] else {},
        "final_sql": state["current_sql"],
        "validation_passed": state["validation_passed"],
        "attempt_count": state["attempt_count"],
        "ex_passed": ex_passed
    }

    return {
        "ex_passed": ex_passed,
        "execution_result": pred_res,
        "audit_record": audit_record
    }


# =====================================================================
# 8. LANGGRAPH COMPILATION & LIVE BACKGROUND MONITOR
# =====================================================================
def validation_router(state: SQLGuardState) -> str:
    if state["validation_passed"]:
        return "execution_gate"
    if state["attempt_count"] < state["max_attempts"]:
        return "repair_agent"
    return "execution_gate"


workflow = StateGraph(SQLGuardState)

workflow.add_node("schema_linker", schema_linker_node)
workflow.add_node("intent_agent", intent_agent_node)
workflow.add_node("generator_decomposer", generator_decomposer_node)
workflow.add_node("hybrid_validator", hybrid_validator_node)
workflow.add_node("repair_agent", repair_agent_node)
workflow.add_node("execution_gate", execution_gate_node)

workflow.add_edge(START, "schema_linker")
workflow.add_edge("schema_linker", "intent_agent")
workflow.add_edge("intent_agent", "generator_decomposer")
workflow.add_edge("generator_decomposer", "hybrid_validator")

workflow.add_conditional_edges(
    "hybrid_validator",
    validation_router,
    {
        "execution_gate": "execution_gate",
        "repair_agent": "repair_agent"
    }
)

workflow.add_edge("repair_agent", "hybrid_validator")
workflow.add_edge("execution_gate", END)

sqlguard_app = workflow.compile()


def process_single_sample(sample: Dict[str, Any], db_map: Dict[str, str]) -> Dict[str, Any]:
    db_id = sample["db_id"]
    db_path = db_map.get(db_id, "")

    initial_state: SQLGuardState = {
        "question": sample["question"],
        "evidence": sample.get("evidence", ""),
        "db_id": db_id,
        "db_path": db_path,
        "gold_sql": sample["SQL"],
        "schema_metadata": "",
        "contract": None,
        "current_sql": "",
        "validation_passed": False,
        "validation_errors": [],
        "validation_error_types": [],
        "attempt_count": 0,
        "max_attempts": 3,
        "initial_failed_sql": "",
        "initial_errors": [],
        "initial_error_types": [],
        "ex_passed": False,
        "execution_result": None,
        "audit_record": {}
    }

    final_state = sqlguard_app.invoke(initial_state)

    with file_lock:
        with open(LOCAL_RESULTS_FILE, "a") as f_out:
            f_out.write(json.dumps(final_state["audit_record"]) + "\n")

    return final_state


def live_progress_logger(stop_event: threading.Event, total_target: int, poll_interval: float = 10.0):
    """Background thread that prints real-time accuracy and recovery metrics."""
    while not stop_event.is_set():
        if os.path.exists(LOCAL_RESULTS_FILE):
            records = []
            with file_lock:
                with open(LOCAL_RESULTS_FILE, "r") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            try:
                                records.append(json.loads(line))
                            except json.JSONDecodeError:
                                continue

            n = len(records)
            if n > 0:
                ex_pass = sum(1 for r in records if r.get("ex_passed", False))
                val_pass = sum(1 for r in records if r.get("validation_passed", False))
                repairs = [r for r in records if r.get("attempt_count", 0) > 0]
                rep_ex = sum(1 for r in repairs if r.get("ex_passed", False))

                pct_done = (n / total_target) * 100
                ex_acc = (ex_pass / n) * 100
                val_rate = (val_pass / n) * 100
                rep_acc = (rep_ex / len(repairs) * 100) if repairs else 0.0

                print(
                    f"\n[LIVE MONITOR] Evaluated: {n}/{total_target} ({pct_done:.1f}%) | "
                    f"EX Acc: {ex_acc:.2f}% | AST Valid: {val_rate:.1f}% | "
                    f"Repairs Recovered: {rep_ex}/{len(repairs)} ({rep_acc:.1f}%)"
                )

        stop_event.wait(poll_interval)


def run_full_bird_benchmark(
    limit_samples: Optional[int] = None,
    dataset_file: str = "dev.json",
    data_dir: str = LOCAL_FULL_DEV_DIR,
    max_workers: int = 6
):
    # 1. Setup local environment & clean previous results
    setup_local_colab_environment()
    clean_and_setup_results_dir()
    global_memory.reload()

    # 2. Locate evaluation JSON
    json_path = None
    for root, _, files in os.walk(data_dir):
        if dataset_file in files:
            json_path = os.path.join(root, dataset_file)
            break

    if not json_path:
        raise FileNotFoundError(f"Could not locate {dataset_file} in '{data_dir}'.")

    with open(json_path, "r") as f:
        full_data = json.load(f)
        samples = full_data[:limit_samples] if limit_samples is not None else full_data

    # 3. Map SQLite databases
    db_map = {}
    for root, _, files in os.walk(data_dir):
        for file in files:
            if file.endswith(".sqlite"):
                db_id = file.replace(".sqlite", "")
                db_map[db_id] = os.path.join(root, file)

    print(f">>> Found {len(db_map)} SQLite databases in {data_dir}.")
    print(f">>> Total dev set samples to evaluate: {len(samples)}")

    passed_semantic_gate = 0
    correct_execution_count = 0
    total_repaired_count = 0

    print(f"\n=================== RUNNING HYBRID SQLGUARD ON FULL BIRD ({len(samples)} SAMPLES, {len(llm_manager.keys)} K2 KEYS) ===================")

    # 4. Start background monitor thread
    stop_monitor_event = threading.Event()
    monitor_thread = threading.Thread(
        target=live_progress_logger,
        args=(stop_monitor_event, len(samples), 15.0),
        daemon=True
    )
    monitor_thread.start()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_sample, sample, db_map) for sample in samples]

        for idx, future in enumerate(tqdm(as_completed(futures), total=len(samples)), 1):
            try:
                final_state = future.result()
                if final_state["validation_passed"]:
                    passed_semantic_gate += 1
                    if final_state["attempt_count"] > 0:
                        total_repaired_count += 1
                if final_state["ex_passed"]:
                    correct_execution_count += 1
            except Exception as e:
                print(f"Sample processing error: {e}")

            if idx % 25 == 0:
                sync_results_to_drive()

    # 5. Stop monitor & perform final sync
    stop_monitor_event.set()
    monitor_thread.join(timeout=1.0)
    sync_results_to_drive()

    total = len(samples)
    print("\n=================== FULL BIRD FINAL BENCHMARK METRICS ===================")
    print(f"Total Samples Evaluated        : {total}")
    print(f"Passed Semantic Contract Gate  : {passed_semantic_gate}/{total} ({passed_semantic_gate/total*100:.1f}%)")
    print(f"Successfully Repaired Queries  : {total_repaired_count}")
    print(f"BIRD Execution Accuracy (EX)   : {correct_execution_count}/{total} ({correct_execution_count/total*100:.1f}%)")
    print(f"Local Results Directory        : {LOCAL_RESULTS_DIR}")
    print(f"Google Drive Results Directory : {get_drive_results_dir()}")


if __name__ == "__main__":
    run_full_bird_benchmark(
        limit_samples=None,  # Evaluates all 1,534 samples
        dataset_file="dev.json",
        data_dir=LOCAL_FULL_DEV_DIR,
        max_workers=6
    )

>>> Loading local SQL generator: seeklhy/codes-3b-bird...


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...


[LIVE MONITOR] Evaluated: 1526/1534 (99.5%) | EX Acc: 52.75% | AST Valid: 99.2% | Repairs Recovered: 330/664 (49.7%)


100%|█████████▉| 1528/1534 [1:28:02<00:23,  3.93s/it]


[LIVE MONITOR] Evaluated: 1528/1534 (99.6%) | EX Acc: 52.75% | AST Valid: 99.2% | Repairs Recovered: 330/664 (49.7%)


100%|██████████| 1534/1534 [1:28:24<00:00,  3.46s/it]


>>> [SYNC SUCCESS] Checkpointed results to Drive: full_dev/results/

=================== FULL BIRD FINAL BENCHMARK METRICS ===================
Total Samples Evaluated        : 1534
Passed Semantic Contract Gate  : 1522/1534 (99.2%)
Successfully Repaired Queries  : 656
BIRD Execution Accuracy (EX)   : 807/1534 (52.6%)
Local Results Directory        : /content/sqlguard_run/full_dev/results
Google Drive Results Directory : /content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627/results


In [ ]:
import os
import re
import json
import time
import shutil
import sqlite3
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Dict, Any, TypedDict
from pydantic import BaseModel, Field
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sqlglot
from sqlglot import parse_one, exp
from openai import OpenAI
from tqdm import tqdm
from langgraph.graph import StateGraph, START, END

# =====================================================================
# 0. DIRECTORY PATHS, FULL_DEV RESOLUTION & RESULTS_7B CLEANING
# =====================================================================
DRIVE_BASE_DIR = "/content/drive/MyDrive/SQLGuard_BIRD"
DRIVE_FULL_DEV_CANDIDATES = [
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev", "dev_20240627"),
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev"),
    os.path.join(DRIVE_BASE_DIR, "bird_data")
]

LOCAL_WORKING_DIR = "/content/sqlguard_run"
LOCAL_FULL_DEV_DIR = os.path.join(LOCAL_WORKING_DIR, "full_dev")
LOCAL_RESULTS_DIR = os.path.join(LOCAL_FULL_DEV_DIR, "results_7b")
LOCAL_RESULTS_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_results_7b.jsonl")
LOCAL_MEMORY_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_failure_memory_7b.json")

os.makedirs(LOCAL_WORKING_DIR, exist_ok=True)
file_lock = threading.Lock()
cache_lock = threading.Lock()
gpu_model_lock = threading.Lock()


def resolve_drive_source_dir() -> str:
    """Locates the directory containing dev.json and dev_databases."""
    for candidate in DRIVE_FULL_DEV_CANDIDATES:
        if os.path.exists(candidate):
            has_json = any(os.path.exists(os.path.join(candidate, f)) for f in ["dev.json", "dev_20240627/dev.json"])
            if has_json:
                return candidate
    return DRIVE_FULL_DEV_CANDIDATES[1]


def get_drive_results_dir() -> str:
    """Returns the target Drive results directory inside full_dev/results_7b/."""
    source_dir = resolve_drive_source_dir()
    return os.path.join(source_dir, "results_7b")


def clean_and_setup_results_dir():
    """Wipes any previous execution artifacts from full_dev/results_7b both locally and on Drive."""
    # 1. Clean local NVMe results directory
    if os.path.exists(LOCAL_RESULTS_DIR):
        print(f">>> Removing previous local 7B results in {LOCAL_RESULTS_DIR}...")
        shutil.rmtree(LOCAL_RESULTS_DIR)
    os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

    # 2. Clean Google Drive results directory
    drive_results_dir = get_drive_results_dir()
    if os.path.exists(drive_results_dir):
        print(f">>> Removing previous Drive 7B results in {drive_results_dir}...")
        shutil.rmtree(drive_results_dir)
    os.makedirs(drive_results_dir, exist_ok=True)

    print(">>> [READY] Fresh full_dev/results_7b directory initialized.")


def setup_local_colab_environment():
    """Copies dataset files from Google Drive to local NVMe storage."""
    source_dir = resolve_drive_source_dir()
    if os.path.exists(source_dir) and not os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Staging full_dev dataset from {source_dir} to local NVMe ({LOCAL_FULL_DEV_DIR})...")
        shutil.copytree(source_dir, LOCAL_FULL_DEV_DIR)
        print(">>> Staging complete!")
    elif os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Local full_dev dataset ready at {LOCAL_FULL_DEV_DIR}.")


def sync_results_to_drive():
    """Atomically syncs incremental results and failure memory back to full_dev/results_7b on Drive."""
    try:
        drive_results_dir = get_drive_results_dir()
        os.makedirs(drive_results_dir, exist_ok=True)
        if os.path.exists(LOCAL_RESULTS_FILE):
            shutil.copy(LOCAL_RESULTS_FILE, os.path.join(drive_results_dir, "sqlguard_results_7b.jsonl"))
        if os.path.exists(LOCAL_MEMORY_FILE):
            shutil.copy(LOCAL_MEMORY_FILE, os.path.join(drive_results_dir, "sqlguard_failure_memory_7b.json"))
        print("\n>>> [SYNC SUCCESS] Checkpointed results to Drive: full_dev/results_7b/")
    except Exception as e:
        print(f"\n>>> [SYNC ERROR] Drive backup failed: {e}")


# =====================================================================
# 1. K2-THINK-V2 ROTATING API MANAGER
# =====================================================================
K2_BASE_URL = os.getenv("K2_BASE_URL", "https://api.k2think.ai/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "MBZUAI-IFM/K2-Think-v2")

API_KEYS = [
    "IFM-93mLmFQS2bfYZZIY",
    "IFM-SQJYZjtO76Kk86de",
    "IFM-BDZ81VHhqLyYn6Ka"
]


def clean_reasoning_output(raw_text: Optional[str]) -> str:
    """Strips <think> tags, markdown fences, and extracts clean JSON/SQL."""
    if not raw_text:
        return ""
    cleaned = re.sub(r"<think>.*?</think>", "", str(raw_text), flags=re.DOTALL).strip()

    if "```json" in cleaned:
        cleaned = cleaned.split("```json")[1].split("```")[0].strip()
    elif "```sql" in cleaned:
        cleaned = cleaned.split("```sql")[1].split("```")[0].strip()
    elif "```" in cleaned:
        cleaned = cleaned.split("```")[1].split("```")[0].strip()

    if not cleaned.startswith("{") and "select" in cleaned.lower():
        select_pos = cleaned.lower().find("select")
        if select_pos != -1:
            cleaned = cleaned[select_pos:].strip()
            cleaned = cleaned.replace("```", "").strip()

    return cleaned


class K2DynamicKeyManager:
    """Round-robin load balancer across active API keys with latency fallback."""
    def __init__(self, keys: List[str], base_url: str):
        self.keys = [k for k in keys if k and not k.startswith("YOUR_")]
        self.base_url = base_url
        self.clients = {k: OpenAI(base_url=self.base_url, api_key=k, timeout=60.0) for k in self.keys}
        self.index = 0
        self.lock = threading.Lock()

    def get_client_and_key(self) -> tuple[OpenAI, str]:
        with self.lock:
            key = self.keys[self.index % len(self.keys)]
            self.index += 1
            return self.clients[key], key

    def rotate_away_from(self, slow_key: str):
        with self.lock:
            if self.keys[self.index % len(self.keys)] == slow_key:
                self.index += 1

    def execute_chat_completion(self, prompt: str, max_retries: int = 2) -> str:
        for attempt in range(max_retries + 1):
            client, active_key = self.get_client_and_key()
            start_time = time.time()
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0
                )
                if time.time() - start_time > 50.0:
                    self.rotate_away_from(active_key)
                content = resp.choices[0].message.content if resp.choices else ""
                return clean_reasoning_output(content)
            except Exception:
                self.rotate_away_from(active_key)
                if attempt == max_retries:
                    return ""
                time.sleep(1.0)
        return ""


llm_manager = K2DynamicKeyManager(API_KEYS, K2_BASE_URL)


# =====================================================================
# 2. LOCAL CODES-7B ENGINE (GPU INFERENCE)
# =====================================================================
class LocalCodeSEngine:
    """Thread-safe GPU inference manager for CodeS-7B[cite: 1]."""
    def __init__(self, model_id: str = "seeklhy/codes-7b-bird"):
        print(f">>> Loading local SQL generator: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        print(">>> CodeS-7B loaded successfully onto GPU.")

    def generate_sql(self, schema_str: str, question: str, evidence: str, contract_guidance: str) -> str:
        prompt = (
            f"-- Database Schema:\n{schema_str}\n"
            f"-- Question: {question}\n"
            f"-- Evidence: {evidence}\n"
            f"-- Semantic Contract Guidance: {contract_guidance}\n"
            f"SELECT"
        )
        with gpu_model_lock:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=160,
                    pad_token_id=self.tokenizer.eos_token_id,
                    do_sample=False,
                    num_beams=4
                )
            generated = "SELECT" + self.tokenizer.decode(
                output_ids[0][inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )
        return clean_reasoning_output(generated)


codes_engine = LocalCodeSEngine()


# =====================================================================
# 3. THREAD-SAFE PERSISTENT FAILURE MEMORY
# =====================================================================
class PersistentFailureMemory:
    """Maintains an on-disk few-shot repository of repaired SQL patterns inside full_dev/results_7b/."""
    def __init__(self, memory_filepath: str = LOCAL_MEMORY_FILE):
        self.filepath = memory_filepath
        self.memory: List[Dict[str, Any]] = []
        self.reload()

    def reload(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r") as f:
                    self.memory = json.load(f)
            except Exception:
                self.memory = []
        else:
            self.memory = []

    def record_repair(self, question: str, failed_sql: str, error_feedback: str, fixed_sql: str, error_types: List[str]):
        entry = {
            "question": question,
            "failed_sql": failed_sql,
            "error_feedback": error_feedback,
            "fixed_sql": fixed_sql,
            "error_types": error_types
        }
        with file_lock:
            self.memory.append(entry)
            with open(self.filepath, "w") as f:
                json.dump(self.memory, f, indent=2)

    def retrieve_similar_repairs(self, current_errors: List[str], max_examples: int = 2) -> str:
        with file_lock:
            if not self.memory:
                return ""
            mem_snapshot = list(self.memory)

        retrieved = []
        for entry in reversed(mem_snapshot):
            for err in current_errors:
                if any(err_type.lower() in err.lower() for err_type in entry.get("error_types", [])):
                    retrieved.append(entry)
                    break
            if len(retrieved) >= max_examples:
                break

        if not retrieved:
            retrieved = mem_snapshot[-max_examples:]

        formatted_cases = []
        for idx, item in enumerate(retrieved, 1):
            formatted_cases.append(
                f"[Past Repair Example #{idx}]\n"
                f"Question: {item['question']}\n"
                f"Failed Query: {item['failed_sql']}\n"
                f"Errors: {item['error_feedback']}\n"
                f"Corrected SQL: {item['fixed_sql']}"
            )
        return "\n\n".join(formatted_cases)


global_memory = PersistentFailureMemory()


# =====================================================================
# 4. 7-DIMENSIONAL SEMANTIC CONTRACT SCHEMA (Γ)
# =====================================================================
class TargetProjection(BaseModel):
    entity: str = Field(description="Summary of target projection")
    output_columns: List[str] = Field(default_factory=list, description="Projected expressions")
    granularity: Optional[str] = Field(default=None, description="Granularity level")

class SchemaLinks(BaseModel):
    required_tables: List[str] = Field(default_factory=list, description="Required tables")
    required_columns: List[str] = Field(default_factory=list, description="Required columns")
    join_keys: List[str] = Field(default_factory=list, description="Join paths")

class Analytics(BaseModel):
    aggregations: List[str] = Field(default_factory=list, description="COUNT, AVG, SUM, MIN, MAX")
    group_by: List[str] = Field(default_factory=list, description="Grouping columns or date slice expressions")
    having: List[str] = Field(default_factory=list, description="HAVING conditions")

class RankingCardinality(BaseModel):
    order_by: List[str] = Field(default_factory=list, description="Sort expressions")
    direction: Optional[str] = Field(default=None, description="ASC or DESC")
    limit: Optional[int] = Field(default=None, description="LIMIT top-k cap")

class SemanticContract(BaseModel):
    target_projection: TargetProjection
    schema_links: SchemaLinks
    predicates: List[str] = Field(default_factory=list, description="WHERE filters preserving INTEGER affinity")
    analytics: Analytics
    ranking_cardinality: RankingCardinality
    read_only: bool = Field(default=True, description="Strict read-only safety flag")
    ambiguity_flag: bool = Field(default=False, description="Ambiguity status")


# =====================================================================
# 5. COMPACT SCHEMA EXTRACTOR & CACHING
# =====================================================================
SCHEMA_CACHE: Dict[str, str] = {}


def extract_compact_schema(db_path: Optional[str]) -> str:
    """Builds a token-efficient, type-annotated SQLite schema description."""
    if not db_path or not os.path.exists(db_path):
        return "Schema unavailable."

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%';")
        tables = [r[0] for r in cursor.fetchall()]

        schema_lines = []
        for table_name in tables:
            cursor.execute(f"PRAGMA table_info('{table_name}');")
            cols = cursor.fetchall()
            col_desc = [f"{c[1]} ({c[2].upper() or 'TEXT'})" for c in cols]
            schema_lines.append(f"TABLE {table_name} (\n  " + ", ".join(col_desc) + "\n)")

            cursor.execute(f"PRAGMA foreign_key_list('{table_name}');")
            for fk in cursor.fetchall():
                schema_lines.append(f"-- FK: {table_name}.{fk[3]} -> {fk[2]}.{fk[4]}")

            samples = []
            sampled_count = 0
            for col in cols:
                if sampled_count >= 4:
                    break
                col_name = col[1]
                cursor.execute(f"SELECT DISTINCT \"{col_name}\" FROM \"{table_name}\" WHERE \"{col_name}\" IS NOT NULL LIMIT 3;")
                vals = [r[0] for r in cursor.fetchall() if r[0] is not None]
                if vals:
                    samples.append(f"{col_name}: {vals}")
                    sampled_count += 1
            if samples:
                schema_lines.append(f"-- [{table_name} Samples]: " + " | ".join(samples))

        conn.close()
        return "\n".join(schema_lines)
    except Exception as e:
        return f"Error reading schema: {e}"


def get_cached_schema(db_id: str, db_path: Optional[str]) -> str:
    with cache_lock:
        if db_id in SCHEMA_CACHE:
            return SCHEMA_CACHE[db_id]

    compact_schema = extract_compact_schema(db_path)
    with cache_lock:
        SCHEMA_CACHE[db_id] = compact_schema
    return compact_schema


# =====================================================================
# 6. NORMALIZED HYBRID AST VALIDATOR (sqlglot)
# =====================================================================
class SQLGuardValidator:
    def validate(self, sql: str, contract: SemanticContract) -> Dict[str, Any]:
        errors = []
        error_types = []

        if not sql or not sql.strip():
            return {"passed": False, "errors": ["Generated SQL was empty."], "error_types": ["Syntax"]}

        try:
            parsed = parse_one(sql, read="sqlite")
        except Exception as e:
            return {"passed": False, "errors": [f"AST Parse Error: {str(e)}"], "error_types": ["Syntax"]}

        if not isinstance(parsed, exp.Select):
            return {
                "passed": False,
                "errors": ["Unit Test [V_safety] Failed: Non-SELECT operation blocked."],
                "error_types": ["Safety"]
            }

        query_tables = {t.name.lower().strip("`'\"[] ") for t in parsed.find_all(exp.Table)}
        query_columns_bare = {c.name.lower().strip("`'\"[] ") for c in parsed.find_all(exp.Column)}

        # 1. Table Verification
        for req_t in contract.schema_links.required_tables:
            req_t_clean = req_t.lower().strip("`'\"[] ")
            if req_t_clean and req_t_clean not in query_tables:
                errors.append(f"Unit Test [Schema Table] Failed: Required table '{req_t}' missing.")
                error_types.append("SchemaTable")

        # 2. Column Verification (Bare vs Qualified)
        for req_c in contract.schema_links.required_columns:
            req_clean = req_c.lower().strip("`'\"[] ")
            req_bare = req_clean.split(".")[-1].strip("`'\"[] ")
            if req_bare and req_bare not in query_columns_bare:
                errors.append(f"Unit Test [Schema Column] Failed: Required column '{req_c}' missing.")
                error_types.append("SchemaColumn")

        # 3. Aggregations
        if contract.analytics.aggregations:
            ast_funcs = set()
            if parsed.find(exp.Count): ast_funcs.add("count")
            if parsed.find(exp.Sum): ast_funcs.add("sum")
            if parsed.find(exp.Avg): ast_funcs.add("avg")
            if parsed.find(exp.Max): ast_funcs.add("max")
            if parsed.find(exp.Min): ast_funcs.add("min")

            for req_agg in contract.analytics.aggregations:
                req_clean = req_agg.lower().strip()
                matched = any(kw in req_clean and kw in ast_funcs for kw in ["count", "sum", "avg", "max", "min"])
                if not matched:
                    errors.append(f"Unit Test [Aggregation] Failed: Missing required function '{req_agg}'.")
                    error_types.append("Aggregation")

        # 4. Predicates
        if contract.predicates:
            has_where = parsed.find(exp.Where) is not None
            has_having = parsed.find(exp.Having) is not None
            if not (has_where or has_having):
                errors.append("Unit Test [Predicate] Failed: Filters specified in contract but WHERE/HAVING missing.")
                error_types.append("Predicate")

        # 5. Group By
        if contract.analytics.group_by and not parsed.find(exp.Group):
            errors.append("Unit Test [GroupBy] Failed: Contract specifies grouping but GROUP BY clause missing.")
            error_types.append("GroupBy")

        # 6. Order By & Limit
        if contract.ranking_cardinality.direction and not parsed.find(exp.Order):
            errors.append("Unit Test [Ranking] Failed: Contract specifies sort order but ORDER BY clause missing.")
            error_types.append("Ranking")

        if contract.ranking_cardinality.limit is not None and not parsed.find(exp.Limit):
            errors.append("Unit Test [Limit] Failed: Contract specifies top-k cap but LIMIT clause missing.")
            error_types.append("Limit")

        return {
            "passed": len(errors) == 0,
            "errors": errors,
            "error_types": list(set(error_types))
        }


# =====================================================================
# 7. LANGGRAPH WORKFLOW NODES
# =====================================================================
class SQLGuardState(TypedDict):
    question: str
    evidence: str
    db_id: str
    db_path: str
    gold_sql: str
    schema_metadata: str
    contract: Optional[SemanticContract]
    current_sql: str
    validation_passed: bool
    validation_errors: List[str]
    validation_error_types: List[str]
    attempt_count: int
    max_attempts: int
    initial_failed_sql: str
    initial_errors: List[str]
    initial_error_types: List[str]
    ex_passed: bool
    execution_result: Optional[List[Any]]
    audit_record: Dict[str, Any]


def schema_linker_node(state: SQLGuardState) -> Dict[str, Any]:
    schema_meta = get_cached_schema(state["db_id"], state["db_path"])
    return {"schema_metadata": schema_meta}


def intent_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    prompt = f"""You are the Intent Agent for SQLGuard. Extract a 7-dimensional Semantic Contract as a JSON object.

### MANDATORY INSTRUCTIONS:
1. DOMAIN EVIDENCE GROUNDING: Strictly follow domain evidence. If evidence indicates date slicing (e.g. SUBSTR(Date, 5, 2) for month, SUBSTR(Date, 1, 4) for year), use that exact expression in output_columns and group_by.
2. SQLITE TYPE COMPLIANCE: If column type is INTEGER (e.g. Date 201301), do NOT wrap numeric numbers in single quotes (use `Date BETWEEN 201301 AND 201312`).
3. PROJECTION PRECISION: Output ONLY the requested attribute or expression in output_columns.

### BENCHMARK EVALUATION GROUNDING RULES:
1. NAME PROJECTION: When asked for a person's name or full name, project two separate columns `first_name, last_name` (or `forename, surname`). DO NOT concatenate with `|| ' ' ||`.
2. CASE-INSENSITIVE TEXT FILTERS: For string equality checks in WHERE clauses, use `COLLATE NOCASE` or `LIKE` (e.g. `Segment = 'Discount' COLLATE NOCASE`).
3. PROJECTION MINIMALISM: Project ONLY the exact attribute requested. Do not include extra tie-breaker columns or IDs in SELECT unless explicitly requested.
4. NULL-SAFE SUMS: Always provide `ELSE 0` in conditional aggregation (e.g., `SUM(CASE WHEN condition THEN val ELSE 0 END)`).

Schema:
{state["schema_metadata"]}

User Question: {state["question"]}
Domain Evidence: {state["evidence"]}

JSON Schema:
{json.dumps(SemanticContract.model_json_schema())}

Output ONLY the raw JSON object."""

    raw_json = llm_manager.execute_chat_completion(prompt)
    try:
        contract = SemanticContract.model_validate_json(raw_json)
    except Exception:
        contract = SemanticContract(
            target_projection=TargetProjection(entity=state["question"]),
            schema_links=SchemaLinks(),
            analytics=Analytics(),
            ranking_cardinality=RankingCardinality()
        )
    return {"contract": contract}


def generator_decomposer_node(state: SQLGuardState) -> Dict[str, Any]:
    contract_str = ""
    if state["contract"]:
        contract_dict = state["contract"].model_dump(exclude_none=True)
        contract_str = json.dumps(contract_dict)

    generated_sql = codes_engine.generate_sql(
        schema_str=state["schema_metadata"],
        question=state["question"],
        evidence=state["evidence"],
        contract_guidance=contract_str
    )
    return {"current_sql": generated_sql}


def hybrid_validator_node(state: SQLGuardState) -> Dict[str, Any]:
    validator = SQLGuardValidator()
    val = validator.validate(state["current_sql"], state["contract"])

    updates = {
        "validation_passed": val["passed"],
        "validation_errors": val["errors"],
        "validation_error_types": val.get("error_types", [])
    }

    if not val["passed"] and state["attempt_count"] == 0:
        updates["initial_failed_sql"] = state["current_sql"]
        updates["initial_errors"] = val["errors"]
        updates["initial_error_types"] = val.get("error_types", [])

    return updates


def repair_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    memory_ctx = global_memory.retrieve_similar_repairs(state["validation_errors"])
    contract_json = state["contract"].model_dump_json() if state["contract"] else "{}"

    prompt = f"""Repair the following SQLite query to satisfy the Semantic Contract and Domain Evidence.

Schema:
{state["schema_metadata"]}

Question: {state["question"]}
Evidence: {state["evidence"]}
Semantic Contract: {contract_json}

[FAILED CANDIDATE SQL]:
{state["current_sql"]}

[CONTRACT VALIDATION ERRORS]:
{chr(10).join(state["validation_errors"])}
"""
    if memory_ctx:
        prompt += f"\n[SIMILAR PAST SUCCESSFUL REPAIRS]:\n{memory_ctx}\n"

    prompt += "\nOutput raw repaired SQL inside a ```sql codeblock."

    repaired_sql = llm_manager.execute_chat_completion(prompt)
    if not repaired_sql:
        repaired_sql = state["current_sql"]

    return {
        "current_sql": repaired_sql,
        "attempt_count": state["attempt_count"] + 1
    }


def execution_gate_node(state: SQLGuardState) -> Dict[str, Any]:
    ex_passed = False
    pred_res = None

    if state["validation_passed"] and state["db_path"] and os.path.exists(state["db_path"]):
        conn = sqlite3.connect(state["db_path"])
        cursor = conn.cursor()
        try:
            cursor.execute(state["current_sql"])
            pred_res = cursor.fetchall()

            cursor.execute(state["gold_sql"])
            gold_res = cursor.fetchall()

            ex_passed = (pred_res == gold_res or set(pred_res) == set(gold_res))
        except Exception:
            ex_passed = False
        finally:
            conn.close()

    if state["validation_passed"] and state["attempt_count"] > 0 and state["initial_failed_sql"]:
        global_memory.record_repair(
            question=state["question"],
            failed_sql=state["initial_failed_sql"],
            error_feedback="\n".join(state["initial_errors"]),
            fixed_sql=state["current_sql"],
            error_types=state["initial_error_types"]
        )

    audit_record = {
        "question": state["question"],
        "db_id": state["db_id"],
        "contract": state["contract"].model_dump() if state["contract"] else {},
        "final_sql": state["current_sql"],
        "validation_passed": state["validation_passed"],
        "attempt_count": state["attempt_count"],
        "ex_passed": ex_passed
    }

    return {
        "ex_passed": ex_passed,
        "execution_result": pred_res,
        "audit_record": audit_record
    }


# =====================================================================
# 8. LANGGRAPH COMPILATION & LIVE BACKGROUND MONITOR
# =====================================================================
def validation_router(state: SQLGuardState) -> str:
    if state["validation_passed"]:
        return "execution_gate"
    if state["attempt_count"] < state["max_attempts"]:
        return "repair_agent"
    return "execution_gate"


workflow = StateGraph(SQLGuardState)

workflow.add_node("schema_linker", schema_linker_node)
workflow.add_node("intent_agent", intent_agent_node)
workflow.add_node("generator_decomposer", generator_decomposer_node)
workflow.add_node("hybrid_validator", hybrid_validator_node)
workflow.add_node("repair_agent", repair_agent_node)
workflow.add_node("execution_gate", execution_gate_node)

workflow.add_edge(START, "schema_linker")
workflow.add_edge("schema_linker", "intent_agent")
workflow.add_edge("intent_agent", "generator_decomposer")
workflow.add_edge("generator_decomposer", "hybrid_validator")

workflow.add_conditional_edges(
    "hybrid_validator",
    validation_router,
    {
        "execution_gate": "execution_gate",
        "repair_agent": "repair_agent"
    }
)

workflow.add_edge("repair_agent", "hybrid_validator")
workflow.add_edge("execution_gate", END)

sqlguard_app = workflow.compile()


def process_single_sample(sample: Dict[str, Any], db_map: Dict[str, str]) -> Dict[str, Any]:
    db_id = sample["db_id"]
    db_path = db_map.get(db_id, "")

    initial_state: SQLGuardState = {
        "question": sample["question"],
        "evidence": sample.get("evidence", ""),
        "db_id": db_id,
        "db_path": db_path,
        "gold_sql": sample["SQL"],
        "schema_metadata": "",
        "contract": None,
        "current_sql": "",
        "validation_passed": False,
        "validation_errors": [],
        "validation_error_types": [],
        "attempt_count": 0,
        "max_attempts": 3,
        "initial_failed_sql": "",
        "initial_errors": [],
        "initial_error_types": [],
        "ex_passed": False,
        "execution_result": None,
        "audit_record": {}
    }

    final_state = sqlguard_app.invoke(initial_state)

    with file_lock:
        with open(LOCAL_RESULTS_FILE, "a") as f_out:
            f_out.write(json.dumps(final_state["audit_record"]) + "\n")

    return final_state


def live_progress_logger(stop_event: threading.Event, total_target: int, poll_interval: float = 10.0):
    """Background thread that prints real-time accuracy and recovery metrics for 7B run."""
    while not stop_event.is_set():
        if os.path.exists(LOCAL_RESULTS_FILE):
            records = []
            with file_lock:
                with open(LOCAL_RESULTS_FILE, "r") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            try:
                                records.append(json.loads(line))
                            except json.JSONDecodeError:
                                continue

            n = len(records)
            if n > 0:
                ex_pass = sum(1 for r in records if r.get("ex_passed", False))
                val_pass = sum(1 for r in records if r.get("validation_passed", False))
                repairs = [r for r in records if r.get("attempt_count", 0) > 0]
                rep_ex = sum(1 for r in repairs if r.get("ex_passed", False))

                pct_done = (n / total_target) * 100
                ex_acc = (ex_pass / n) * 100
                val_rate = (val_pass / n) * 100
                rep_acc = (rep_ex / len(repairs) * 100) if repairs else 0.0

                print(
                    f"\n[LIVE 7B MONITOR] Evaluated: {n}/{total_target} ({pct_done:.1f}%) | "
                    f"EX Acc: {ex_acc:.2f}% | AST Valid: {val_rate:.1f}% | "
                    f"Repairs Recovered: {rep_ex}/{len(repairs)} ({rep_acc:.1f}%)"
                )

        stop_event.wait(poll_interval)


def run_full_bird_benchmark(
    limit_samples: Optional[int] = None,
    dataset_file: str = "dev.json",
    data_dir: str = LOCAL_FULL_DEV_DIR,
    max_workers: int = 6
):
    # 1. Setup local environment & clean previous results in results_7b
    setup_local_colab_environment()
    clean_and_setup_results_dir()
    global_memory.reload()

    # 2. Locate evaluation JSON
    json_path = None
    for root, _, files in os.walk(data_dir):
        if dataset_file in files:
            json_path = os.path.join(root, dataset_file)
            break

    if not json_path:
        raise FileNotFoundError(f"Could not locate {dataset_file} in '{data_dir}'.")

    with open(json_path, "r") as f:
        full_data = json.load(f)
        samples = full_data[:limit_samples] if limit_samples is not None else full_data

    # 3. Map SQLite databases
    db_map = {}
    for root, _, files in os.walk(data_dir):
        for file in files:
            if file.endswith(".sqlite"):
                db_id = file.replace(".sqlite", "")
                db_map[db_id] = os.path.join(root, file)

    print(f">>> Found {len(db_map)} SQLite databases in {data_dir}.")
    print(f">>> Total dev set samples to evaluate: {len(samples)}")

    passed_semantic_gate = 0
    correct_execution_count = 0
    total_repaired_count = 0

    print(f"\n=================== RUNNING HYBRID SQLGUARD WITH CODES-7B ({len(samples)} SAMPLES, {len(llm_manager.keys)} K2 KEYS) ===================")

    # 4. Start background monitor thread
    stop_monitor_event = threading.Event()
    monitor_thread = threading.Thread(
        target=live_progress_logger,
        args=(stop_monitor_event, len(samples), 15.0),
        daemon=True
    )
    monitor_thread.start()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_sample, sample, db_map) for sample in samples]

        for idx, future in enumerate(tqdm(as_completed(futures), total=len(samples)), 1):
            try:
                final_state = future.result()
                if final_state["validation_passed"]:
                    passed_semantic_gate += 1
                    if final_state["attempt_count"] > 0:
                        total_repaired_count += 1
                if final_state["ex_passed"]:
                    correct_execution_count += 1
            except Exception as e:
                print(f"Sample processing error: {e}")

            if idx % 25 == 0:
                sync_results_to_drive()

    # 5. Stop monitor & perform final sync
    stop_monitor_event.set()
    monitor_thread.join(timeout=1.0)
    sync_results_to_drive()

    total = len(samples)
    print("\n=================== FULL BIRD FINAL BENCHMARK METRICS (7B) ===================")
    print(f"Total Samples Evaluated        : {total}")
    print(f"Passed Semantic Contract Gate  : {passed_semantic_gate}/{total} ({passed_semantic_gate/total*100:.1f}%)")
    print(f"Successfully Repaired Queries  : {total_repaired_count}")
    print(f"BIRD Execution Accuracy (EX)   : {correct_execution_count}/{total} ({correct_execution_count/total*100:.1f}%)")
    print(f"Local Results Directory        : {LOCAL_RESULTS_DIR}")
    print(f"Google Drive Results Directory : {get_drive_results_dir()}")


if __name__ == "__main__":
    run_full_bird_benchmark(
        limit_samples=None,  # Evaluates all 1,534 samples
        dataset_file="dev.json",
        data_dir=LOCAL_FULL_DEV_DIR,
        max_workers=6
    )

>>> Loading local SQL generator: seeklhy/codes-7b-bird...


config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...


[LIVE 7B MONITOR] Evaluated: 1531/1534 (99.8%) | EX Acc: 54.67% | AST Valid: 98.5% | Repairs Recovered: 283/583 (48.5%)


100%|█████████▉| 1533/1534 [2:19:24<00:05,  5.19s/it]


[LIVE 7B MONITOR] Evaluated: 1533/1534 (99.9%) | EX Acc: 54.60% | AST Valid: 98.5% | Repairs Recovered: 283/584 (48.5%)

[LIVE 7B MONITOR] Evaluated: 1533/1534 (99.9%) | EX Acc: 54.60% | AST Valid: 98.5% | Repairs Recovered: 283/584 (48.5%)


100%|██████████| 1534/1534 [2:20:00<00:00,  5.48s/it]


>>> [SYNC SUCCESS] Checkpointed results to Drive: full_dev/results_7b/

=================== FULL BIRD FINAL BENCHMARK METRICS (7B) ===================
Total Samples Evaluated        : 1534
Passed Semantic Contract Gate  : 1511/1534 (98.5%)
Successfully Repaired Queries  : 562
BIRD Execution Accuracy (EX)   : 837/1534 (54.6%)
Local Results Directory        : /content/sqlguard_run/full_dev/results_7b
Google Drive Results Directory : /content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627/results_7b


In [ ]:
import os
import re
import json
import time
import shutil
import sqlite3
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Dict, Any, TypedDict
from pydantic import BaseModel, Field
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sqlglot
from sqlglot import parse_one, exp
from openai import OpenAI
from tqdm import tqdm
from langgraph.graph import StateGraph, START, END

# =====================================================================
# 0. DIRECTORY PATHS, FULL_DEV RESOLUTION & RESULTS_7B CLEANING
# =====================================================================
DRIVE_BASE_DIR = "/content/drive/MyDrive/SQLGuard_BIRD"
DRIVE_FULL_DEV_CANDIDATES = [
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev", "dev_20240627"),
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev"),
    os.path.join(DRIVE_BASE_DIR, "bird_data")
]

LOCAL_WORKING_DIR = "/content/sqlguard_run"
LOCAL_FULL_DEV_DIR = os.path.join(LOCAL_WORKING_DIR, "full_dev")
LOCAL_RESULTS_DIR = os.path.join(LOCAL_FULL_DEV_DIR, "results_7b_with_evidence")
LOCAL_RESULTS_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_results_7b_with_evidence.jsonl")
LOCAL_MEMORY_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_failure_memory_7b_with_evidence.json")

os.makedirs(LOCAL_WORKING_DIR, exist_ok=True)
file_lock = threading.Lock()
cache_lock = threading.Lock()
gpu_model_lock = threading.Lock()


def resolve_drive_source_dir() -> str:
    """Locates the directory containing dev.json and dev_databases."""
    for candidate in DRIVE_FULL_DEV_CANDIDATES:
        if os.path.exists(candidate):
            has_json = any(os.path.exists(os.path.join(candidate, f)) for f in ["dev.json", "dev_20240627/dev.json"])
            if has_json:
                return candidate
    return DRIVE_FULL_DEV_CANDIDATES[1]


def get_drive_results_dir() -> str:
    """Returns the target Drive results directory inside full_dev/results_7b_with_evidence/."""
    source_dir = resolve_drive_source_dir()
    return os.path.join(source_dir, "results_7b_with_evidence")


def clean_and_setup_results_dir():
    """Wipes any previous execution artifacts from full_dev/results_7b_with_evidence both locally and on Drive."""
    # 1. Clean local NVMe results directory
    if os.path.exists(LOCAL_RESULTS_DIR):
        print(f">>> Removing previous local 7B results in {LOCAL_RESULTS_DIR}...")
        shutil.rmtree(LOCAL_RESULTS_DIR)
    os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

    # 2. Clean Google Drive results directory
    drive_results_dir = get_drive_results_dir()
    if os.path.exists(drive_results_dir):
        print(f">>> Removing previous Drive 7B results in {drive_results_dir}...")
        shutil.rmtree(drive_results_dir)
    os.makedirs(drive_results_dir, exist_ok=True)

    print(">>> [READY] Fresh full_dev/results_7b_with_evidence directory initialized.")


def setup_local_colab_environment():
    """Copies dataset files from Google Drive to local NVMe storage."""
    source_dir = resolve_drive_source_dir()
    if os.path.exists(source_dir) and not os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Staging full_dev dataset from {source_dir} to local NVMe ({LOCAL_FULL_DEV_DIR})...")
        shutil.copytree(source_dir, LOCAL_FULL_DEV_DIR)
        print(">>> Staging complete!")
    elif os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Local full_dev dataset ready at {LOCAL_FULL_DEV_DIR}.")


def sync_results_to_drive():
    """Atomically syncs incremental results and failure memory back to full_dev/results_7b_with_evidence on Drive."""
    try:
        drive_results_dir = get_drive_results_dir()
        os.makedirs(drive_results_dir, exist_ok=True)
        if os.path.exists(LOCAL_RESULTS_FILE):
            shutil.copy(LOCAL_RESULTS_FILE, os.path.join(drive_results_dir, "sqlguard_results_7b_with_evidence.jsonl"))
        if os.path.exists(LOCAL_MEMORY_FILE):
            shutil.copy(LOCAL_MEMORY_FILE, os.path.join(drive_results_dir, "sqlguard_failure_memory_7b_with_evidence.json"))
        print("\n>>> [SYNC SUCCESS] Checkpointed results to Drive: full_dev/results_7b_with_evidence/")
    except Exception as e:
        print(f"\n>>> [SYNC ERROR] Drive backup failed: {e}")


# =====================================================================
# 1. K2-THINK-V2 ROTATING API MANAGER
# =====================================================================
K2_BASE_URL = os.getenv("K2_BASE_URL", "https://api.k2think.ai/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "MBZUAI-IFM/K2-Think-v2")

API_KEYS = [
    "IFM-93mLmFQS2bfYZZIY",
    "IFM-SQJYZjtO76Kk86de",
    "IFM-BDZ81VHhqLyYn6Ka"
]


def clean_reasoning_output(raw_text: Optional[str]) -> str:
    """Strips <think> tags, markdown fences, and extracts clean JSON/SQL."""
    if not raw_text:
        return ""
    cleaned = re.sub(r"<think>.*?</think>", "", str(raw_text), flags=re.DOTALL).strip()

    if "```json" in cleaned:
        cleaned = cleaned.split("```json")[1].split("```")[0].strip()
    elif "```sql" in cleaned:
        cleaned = cleaned.split("```sql")[1].split("```")[0].strip()
    elif "```" in cleaned:
        cleaned = cleaned.split("```")[1].split("```")[0].strip()

    if not cleaned.startswith("{") and "select" in cleaned.lower():
        select_pos = cleaned.lower().find("select")
        if select_pos != -1:
            cleaned = cleaned[select_pos:].strip()
            cleaned = cleaned.replace("```", "").strip()

    return cleaned


class K2DynamicKeyManager:
    """Round-robin load balancer across active API keys with latency fallback."""
    def __init__(self, keys: List[str], base_url: str):
        self.keys = [k for k in keys if k and not k.startswith("YOUR_")]
        self.base_url = base_url
        self.clients = {k: OpenAI(base_url=self.base_url, api_key=k, timeout=60.0) for k in self.keys}
        self.index = 0
        self.lock = threading.Lock()

    def get_client_and_key(self) -> tuple[OpenAI, str]:
        with self.lock:
            key = self.keys[self.index % len(self.keys)]
            self.index += 1
            return self.clients[key], key

    def rotate_away_from(self, slow_key: str):
        with self.lock:
            if self.keys[self.index % len(self.keys)] == slow_key:
                self.index += 1

    def execute_chat_completion(self, prompt: str, max_retries: int = 2) -> str:
        for attempt in range(max_retries + 1):
            client, active_key = self.get_client_and_key()
            start_time = time.time()
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0
                )
                if time.time() - start_time > 50.0:
                    self.rotate_away_from(active_key)
                content = resp.choices[0].message.content if resp.choices else ""
                return clean_reasoning_output(content)
            except Exception:
                self.rotate_away_from(active_key)
                if attempt == max_retries:
                    return ""
                time.sleep(1.0)
        return ""


llm_manager = K2DynamicKeyManager(API_KEYS, K2_BASE_URL)


# =====================================================================
# 2. LOCAL CODES-7B ENGINE (WITH EVIDENCE CHECKPOINT & SFT FORMAT)
# =====================================================================
class LocalCodeSEngine:
    """Thread-safe GPU inference manager for CodeS-7B with evidence conditioning."""
    def __init__(self, model_id: str = "seeklhy/codes-7b-bird-with-evidence"):
        print(f">>> Loading local SQL generator: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        print(">>> CodeS-7B (With Evidence) loaded successfully onto GPU.")

    def generate_sql(self, schema_str: str, question: str, evidence: str) -> str:
        """Constructs the canonical CodeS SFT prompt format to preserve native accuracy."""
        prompt = (
            f"Given the database schema, you need to translate the natural language question into SQL query.\n\n"
            f"[Database schema]\n{schema_str}\n\n"
            f"[Question]\n{question}\n\n"
            f"[Evidence]\n{evidence}\n\n"
            f"[SQL]\nSELECT"
        )
        with gpu_model_lock:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=160,
                    pad_token_id=self.tokenizer.eos_token_id,
                    do_sample=False,
                    num_beams=4
                )
            generated = "SELECT" + self.tokenizer.decode(
                output_ids[0][inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )
        return clean_reasoning_output(generated)


codes_engine = LocalCodeSEngine()


# =====================================================================
# 3. THREAD-SAFE PERSISTENT FAILURE MEMORY
# =====================================================================
class PersistentFailureMemory:
    """Maintains an on-disk few-shot repository of repaired SQL patterns inside full_dev/results_7b_with_evidence/."""
    def __init__(self, memory_filepath: str = LOCAL_MEMORY_FILE):
        self.filepath = memory_filepath
        self.memory: List[Dict[str, Any]] = []
        self.reload()

    def reload(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r") as f:
                    self.memory = json.load(f)
            except Exception:
                self.memory = []
        else:
            self.memory = []

    def record_repair(self, question: str, failed_sql: str, error_feedback: str, fixed_sql: str, error_types: List[str]):
        entry = {
            "question": question,
            "failed_sql": failed_sql,
            "error_feedback": error_feedback,
            "fixed_sql": fixed_sql,
            "error_types": error_types
        }
        with file_lock:
            self.memory.append(entry)
            with open(self.filepath, "w") as f:
                json.dump(self.memory, f, indent=2)

    def retrieve_similar_repairs(self, current_errors: List[str], max_examples: int = 2) -> str:
        with file_lock:
            if not self.memory:
                return ""
            mem_snapshot = list(self.memory)

        retrieved = []
        for entry in reversed(mem_snapshot):
            for err in current_errors:
                if any(err_type.lower() in err.lower() for err_type in entry.get("error_types", [])):
                    retrieved.append(entry)
                    break
            if len(retrieved) >= max_examples:
                break

        if not retrieved:
            retrieved = mem_snapshot[-max_examples:]

        formatted_cases = []
        for idx, item in enumerate(retrieved, 1):
            formatted_cases.append(
                f"[Past Repair Example #{idx}]\n"
                f"Question: {item['question']}\n"
                f"Failed Query: {item['failed_sql']}\n"
                f"Errors: {item['error_feedback']}\n"
                f"Corrected SQL: {item['fixed_sql']}"
            )
        return "\n\n".join(formatted_cases)


global_memory = PersistentFailureMemory()


# =====================================================================
# 4. 7-DIMENSIONAL SEMANTIC CONTRACT SCHEMA (Γ)
# =====================================================================
class TargetProjection(BaseModel):
    entity: str = Field(description="Summary of target projection")
    output_columns: List[str] = Field(default_factory=list, description="Projected expressions")
    granularity: Optional[str] = Field(default=None, description="Granularity level")

class SchemaLinks(BaseModel):
    required_tables: List[str] = Field(default_factory=list, description="Required tables")
    required_columns: List[str] = Field(default_factory=list, description="Required columns")
    join_keys: List[str] = Field(default_factory=list, description="Join paths")

class Analytics(BaseModel):
    aggregations: List[str] = Field(default_factory=list, description="COUNT, AVG, SUM, MIN, MAX")
    group_by: List[str] = Field(default_factory=list, description="Grouping columns or date slice expressions")
    having: List[str] = Field(default_factory=list, description="HAVING conditions")

class RankingCardinality(BaseModel):
    order_by: List[str] = Field(default_factory=list, description="Sort expressions")
    direction: Optional[str] = Field(default=None, description="ASC or DESC")
    limit: Optional[int] = Field(default=None, description="LIMIT top-k cap")

class SemanticContract(BaseModel):
    target_projection: TargetProjection
    schema_links: SchemaLinks
    predicates: List[str] = Field(default_factory=list, description="WHERE filters preserving INTEGER affinity")
    analytics: Analytics
    ranking_cardinality: RankingCardinality
    read_only: bool = Field(default=True, description="Strict read-only safety flag")
    ambiguity_flag: bool = Field(default=False, description="Ambiguity status")


# =====================================================================
# 5. COMPACT SCHEMA EXTRACTOR & CACHING
# =====================================================================
SCHEMA_CACHE: Dict[str, str] = {}


def extract_compact_schema(db_path: Optional[str]) -> str:
    """Builds a token-efficient, type-annotated SQLite schema description."""
    if not db_path or not os.path.exists(db_path):
        return "Schema unavailable."

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%';")
        tables = [r[0] for r in cursor.fetchall()]

        schema_lines = []
        for table_name in tables:
            cursor.execute(f"PRAGMA table_info('{table_name}');")
            cols = cursor.fetchall()
            col_desc = [f"{c[1]} ({c[2].upper() or 'TEXT'})" for c in cols]
            schema_lines.append(f"TABLE {table_name} (\n  " + ", ".join(col_desc) + "\n)")

            cursor.execute(f"PRAGMA foreign_key_list('{table_name}');")
            for fk in cursor.fetchall():
                schema_lines.append(f"-- FK: {table_name}.{fk[3]} -> {fk[2]}.{fk[4]}")

            samples = []
            sampled_count = 0
            for col in cols:
                if sampled_count >= 4:
                    break
                col_name = col[1]
                cursor.execute(f"SELECT DISTINCT \"{col_name}\" FROM \"{table_name}\" WHERE \"{col_name}\" IS NOT NULL LIMIT 3;")
                vals = [r[0] for r in cursor.fetchall() if r[0] is not None]
                if vals:
                    samples.append(f"{col_name}: {vals}")
                    sampled_count += 1
            if samples:
                schema_lines.append(f"-- [{table_name} Samples]: " + " | ".join(samples))

        conn.close()
        return "\n".join(schema_lines)
    except Exception as e:
        return f"Error reading schema: {e}"


def get_cached_schema(db_id: str, db_path: Optional[str]) -> str:
    with cache_lock:
        if db_id in SCHEMA_CACHE:
            return SCHEMA_CACHE[db_id]

    compact_schema = extract_compact_schema(db_path)
    with cache_lock:
        SCHEMA_CACHE[db_id] = compact_schema
    return compact_schema


# =====================================================================
# 6. NORMALIZED HYBRID AST VALIDATOR (sqlglot)
# =====================================================================
class SQLGuardValidator:
    def validate(self, sql: str, contract: SemanticContract) -> Dict[str, Any]:
        errors = []
        error_types = []

        if not sql or not sql.strip():
            return {"passed": False, "errors": ["Generated SQL was empty."], "error_types": ["Syntax"]}

        try:
            parsed = parse_one(sql, read="sqlite")
        except Exception as e:
            return {"passed": False, "errors": [f"AST Parse Error: {str(e)}"], "error_types": ["Syntax"]}

        if not isinstance(parsed, exp.Select):
            return {
                "passed": False,
                "errors": ["Unit Test [V_safety] Failed: Non-SELECT operation blocked."],
                "error_types": ["Safety"]
            }

        query_tables = {t.name.lower().strip("`'\"[] ") for t in parsed.find_all(exp.Table)}
        query_columns_bare = {c.name.lower().strip("`'\"[] ") for c in parsed.find_all(exp.Column)}

        # 1. Table Verification
        for req_t in contract.schema_links.required_tables:
            req_t_clean = req_t.lower().strip("`'\"[] ")
            if req_t_clean and req_t_clean not in query_tables:
                errors.append(f"Unit Test [Schema Table] Failed: Required table '{req_t}' missing.")
                error_types.append("SchemaTable")

        # 2. Column Verification (Bare vs Qualified)
        for req_c in contract.schema_links.required_columns:
            req_clean = req_c.lower().strip("`'\"[] ")
            req_bare = req_clean.split(".")[-1].strip("`'\"[] ")
            if req_bare and req_bare not in query_columns_bare:
                errors.append(f"Unit Test [Schema Column] Failed: Required column '{req_c}' missing.")
                error_types.append("SchemaColumn")

        # 3. Aggregations
        if contract.analytics.aggregations:
            ast_funcs = set()
            if parsed.find(exp.Count): ast_funcs.add("count")
            if parsed.find(exp.Sum): ast_funcs.add("sum")
            if parsed.find(exp.Avg): ast_funcs.add("avg")
            if parsed.find(exp.Max): ast_funcs.add("max")
            if parsed.find(exp.Min): ast_funcs.add("min")

            for req_agg in contract.analytics.aggregations:
                req_clean = req_agg.lower().strip()
                matched = any(kw in req_clean and kw in ast_funcs for kw in ["count", "sum", "avg", "max", "min"])
                if not matched:
                    errors.append(f"Unit Test [Aggregation] Failed: Missing required function '{req_agg}'.")
                    error_types.append("Aggregation")

        # 4. Predicates
        if contract.predicates:
            has_where = parsed.find(exp.Where) is not None
            has_having = parsed.find(exp.Having) is not None
            if not (has_where or has_having):
                errors.append("Unit Test [Predicate] Failed: Filters specified in contract but WHERE/HAVING missing.")
                error_types.append("Predicate")

        # 5. Group By
        if contract.analytics.group_by and not parsed.find(exp.Group):
            errors.append("Unit Test [GroupBy] Failed: Contract specifies grouping but GROUP BY clause missing.")
            error_types.append("GroupBy")

        # 6. Order By & Limit
        if contract.ranking_cardinality.direction and not parsed.find(exp.Order):
            errors.append("Unit Test [Ranking] Failed: Contract specifies sort order but ORDER BY clause missing.")
            error_types.append("Ranking")

        if contract.ranking_cardinality.limit is not None and not parsed.find(exp.Limit):
            errors.append("Unit Test [Limit] Failed: Contract specifies top-k cap but LIMIT clause missing.")
            error_types.append("Limit")

        return {
            "passed": len(errors) == 0,
            "errors": errors,
            "error_types": list(set(error_types))
        }


# =====================================================================
# 7. LANGGRAPH WORKFLOW NODES
# =====================================================================
class SQLGuardState(TypedDict):
    question: str
    evidence: str
    db_id: str
    db_path: str
    gold_sql: str
    schema_metadata: str
    contract: Optional[SemanticContract]
    current_sql: str
    validation_passed: bool
    validation_errors: List[str]
    validation_error_types: List[str]
    attempt_count: int
    max_attempts: int
    initial_failed_sql: str
    initial_errors: List[str]
    initial_error_types: List[str]
    ex_passed: bool
    execution_result: Optional[List[Any]]
    audit_record: Dict[str, Any]


def schema_linker_node(state: SQLGuardState) -> Dict[str, Any]:
    schema_meta = get_cached_schema(state["db_id"], state["db_path"])
    return {"schema_metadata": schema_meta}


def intent_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    prompt = f"""You are the Intent Agent for SQLGuard. Extract a 7-dimensional Semantic Contract as a JSON object.

### MANDATORY INSTRUCTIONS:
1. DOMAIN EVIDENCE GROUNDING: Strictly follow domain evidence. If evidence indicates date slicing (e.g. SUBSTR(Date, 5, 2) for month, SUBSTR(Date, 1, 4) for year), use that exact expression in output_columns and group_by.
2. SQLITE TYPE COMPLIANCE: If column type is INTEGER (e.g. Date 201301), do NOT wrap numeric numbers in single quotes (use `Date BETWEEN 201301 AND 201312`).
3. PROJECTION PRECISION: Output ONLY the requested attribute or expression in output_columns.

### BENCHMARK EVALUATION GROUNDING RULES:
1. NAME PROJECTION: When asked for a person's name or full name, project two separate columns `first_name, last_name` (or `forename, surname`). DO NOT concatenate with `|| ' ' ||`.
2. CASE-INSENSITIVE TEXT FILTERS: For string equality checks in WHERE clauses, use `COLLATE NOCASE` or `LIKE` (e.g. `Segment = 'Discount' COLLATE NOCASE`).
3. PROJECTION MINIMALISM: Project ONLY the exact attribute requested. Do not include extra tie-breaker columns or IDs in SELECT unless explicitly requested.
4. NULL-SAFE SUMS: Always provide `ELSE 0` in conditional aggregation (e.g., `SUM(CASE WHEN condition THEN val ELSE 0 END)`).

Schema:
{state["schema_metadata"]}

User Question: {state["question"]}
Domain Evidence: {state["evidence"]}

JSON Schema:
{json.dumps(SemanticContract.model_json_schema())}

Output ONLY the raw JSON object."""

    raw_json = llm_manager.execute_chat_completion(prompt)
    try:
        contract = SemanticContract.model_validate_json(raw_json)
    except Exception:
        contract = SemanticContract(
            target_projection=TargetProjection(entity=state["question"]),
            schema_links=SchemaLinks(),
            analytics=Analytics(),
            ranking_cardinality=RankingCardinality()
        )
    return {"contract": contract}


def generator_decomposer_node(state: SQLGuardState) -> Dict[str, Any]:
    generated_sql = codes_engine.generate_sql(
        schema_str=state["schema_metadata"],
        question=state["question"],
        evidence=state["evidence"]
    )
    return {"current_sql": generated_sql}


def hybrid_validator_node(state: SQLGuardState) -> Dict[str, Any]:
    validator = SQLGuardValidator()
    val = validator.validate(state["current_sql"], state["contract"])

    updates = {
        "validation_passed": val["passed"],
        "validation_errors": val["errors"],
        "validation_error_types": val.get("error_types", [])
    }

    if not val["passed"] and state["attempt_count"] == 0:
        updates["initial_failed_sql"] = state["current_sql"]
        updates["initial_errors"] = val["errors"]
        updates["initial_error_types"] = val.get("error_types", [])

    return updates


def repair_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    memory_ctx = global_memory.retrieve_similar_repairs(state["validation_errors"])
    contract_json = state["contract"].model_dump_json() if state["contract"] else "{}"

    prompt = f"""Repair the following SQLite query to satisfy the Semantic Contract and Domain Evidence.

Schema:
{state["schema_metadata"]}

Question: {state["question"]}
Evidence: {state["evidence"]}
Semantic Contract: {contract_json}

[FAILED CANDIDATE SQL]:
{state["current_sql"]}

[CONTRACT VALIDATION ERRORS]:
{chr(10).join(state["validation_errors"])}
"""
    if memory_ctx:
        prompt += f"\n[SIMILAR PAST SUCCESSFUL REPAIRS]:\n{memory_ctx}\n"

    prompt += "\nOutput raw repaired SQL inside a ```sql codeblock."

    repaired_sql = llm_manager.execute_chat_completion(prompt)
    if not repaired_sql:
        repaired_sql = state["current_sql"]

    return {
        "current_sql": repaired_sql,
        "attempt_count": state["attempt_count"] + 1
    }


def execution_gate_node(state: SQLGuardState) -> Dict[str, Any]:
    ex_passed = False
    pred_res = None

    if state["validation_passed"] and state["db_path"] and os.path.exists(state["db_path"]):
        conn = sqlite3.connect(state["db_path"], timeout=10.0)
        cursor = conn.cursor()
        try:
            cursor.execute(state["current_sql"])
            pred_res = cursor.fetchall()

            cursor.execute(state["gold_sql"])
            gold_res = cursor.fetchall()

            # Robust float normalization comparison
            def normalize_cell(val):
                if isinstance(val, float):
                    return round(val, 2)
                return val

            def normalize_rows(rows):
                if not rows:
                    return []
                return [tuple(normalize_cell(c) for c in row) for row in rows]

            pred_norm = normalize_rows(pred_res)
            gold_norm = normalize_rows(gold_res)

            ex_passed = (pred_norm == gold_norm or set(pred_norm) == set(gold_norm))
        except Exception:
            ex_passed = False
        finally:
            conn.close()

    if state["validation_passed"] and state["attempt_count"] > 0 and state["initial_failed_sql"]:
        global_memory.record_repair(
            question=state["question"],
            failed_sql=state["initial_failed_sql"],
            error_feedback="\n".join(state["initial_errors"]),
            fixed_sql=state["current_sql"],
            error_types=state["initial_error_types"]
        )

    audit_record = {
        "question": state["question"],
        "db_id": state["db_id"],
        "contract": state["contract"].model_dump() if state["contract"] else {},
        "final_sql": state["current_sql"],
        "validation_passed": state["validation_passed"],
        "attempt_count": state["attempt_count"],
        "ex_passed": ex_passed
    }

    return {
        "ex_passed": ex_passed,
        "execution_result": pred_res,
        "audit_record": audit_record
    }


# =====================================================================
# 8. LANGGRAPH COMPILATION & LIVE BACKGROUND MONITOR
# =====================================================================
def validation_router(state: SQLGuardState) -> str:
    if state["validation_passed"]:
        return "execution_gate"
    if state["attempt_count"] < state["max_attempts"]:
        return "repair_agent"
    return "execution_gate"


workflow = StateGraph(SQLGuardState)

workflow.add_node("schema_linker", schema_linker_node)
workflow.add_node("intent_agent", intent_agent_node)
workflow.add_node("generator_decomposer", generator_decomposer_node)
workflow.add_node("hybrid_validator", hybrid_validator_node)
workflow.add_node("repair_agent", repair_agent_node)
workflow.add_node("execution_gate", execution_gate_node)

workflow.add_edge(START, "schema_linker")
workflow.add_edge("schema_linker", "intent_agent")
workflow.add_edge("intent_agent", "generator_decomposer")
workflow.add_edge("generator_decomposer", "hybrid_validator")

workflow.add_conditional_edges(
    "hybrid_validator",
    validation_router,
    {
        "execution_gate": "execution_gate",
        "repair_agent": "repair_agent"
    }
)

workflow.add_edge("repair_agent", "hybrid_validator")
workflow.add_edge("execution_gate", END)

sqlguard_app = workflow.compile()


def process_single_sample(sample: Dict[str, Any], db_map: Dict[str, str]) -> Dict[str, Any]:
    db_id = sample["db_id"]
    db_path = db_map.get(db_id, "")

    initial_state: SQLGuardState = {
        "question": sample["question"],
        "evidence": sample.get("evidence", ""),
        "db_id": db_id,
        "db_path": db_path,
        "gold_sql": sample["SQL"],
        "schema_metadata": "",
        "contract": None,
        "current_sql": "",
        "validation_passed": False,
        "validation_errors": [],
        "validation_error_types": [],
        "attempt_count": 0,
        "max_attempts": 3,
        "initial_failed_sql": "",
        "initial_errors": [],
        "initial_error_types": [],
        "ex_passed": False,
        "execution_result": None,
        "audit_record": {}
    }

    final_state = sqlguard_app.invoke(initial_state)

    with file_lock:
        with open(LOCAL_RESULTS_FILE, "a") as f_out:
            f_out.write(json.dumps(final_state["audit_record"]) + "\n")

    return final_state


def live_progress_logger(stop_event: threading.Event, total_target: int, poll_interval: float = 10.0):
    """Background thread that prints real-time accuracy and recovery metrics for 7B run."""
    while not stop_event.is_set():
        if os.path.exists(LOCAL_RESULTS_FILE):
            records = []
            with file_lock:
                with open(LOCAL_RESULTS_FILE, "r") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            try:
                                records.append(json.loads(line))
                            except json.JSONDecodeError:
                                continue

            n = len(records)
            if n > 0:
                ex_pass = sum(1 for r in records if r.get("ex_passed", False))
                val_pass = sum(1 for r in records if r.get("validation_passed", False))
                repairs = [r for r in records if r.get("attempt_count", 0) > 0]
                rep_ex = sum(1 for r in repairs if r.get("ex_passed", False))

                pct_done = (n / total_target) * 100
                ex_acc = (ex_pass / n) * 100
                val_rate = (val_pass / n) * 100
                rep_acc = (rep_ex / len(repairs) * 100) if repairs else 0.0

                print(
                    f"\n[LIVE 7B MONITOR] Evaluated: {n}/{total_target} ({pct_done:.1f}%) | "
                    f"EX Acc: {ex_acc:.2f}% | AST Valid: {val_rate:.1f}% | "
                    f"Repairs Recovered: {rep_ex}/{len(repairs)} ({rep_acc:.1f}%)"
                )

        stop_event.wait(poll_interval)


def run_full_bird_benchmark(
    limit_samples: Optional[int] = None,
    dataset_file: str = "dev.json",
    data_dir: str = LOCAL_FULL_DEV_DIR,
    max_workers: int = 6
):
    # 1. Setup local environment & clean previous results in results_7b_with_evidence
    setup_local_colab_environment()
    clean_and_setup_results_dir()
    global_memory.reload()

    # 2. Locate evaluation JSON
    json_path = None
    for root, _, files in os.walk(data_dir):
        if dataset_file in files:
            json_path = os.path.join(root, dataset_file)
            break

    if not json_path:
        raise FileNotFoundError(f"Could not locate {dataset_file} in '{data_dir}'.")

    with open(json_path, "r") as f:
        full_data = json.load(f)
        samples = full_data[:limit_samples] if limit_samples is not None else full_data

    # 3. Map SQLite databases
    db_map = {}
    for root, _, files in os.walk(data_dir):
        for file in files:
            if file.endswith(".sqlite"):
                db_id = file.replace(".sqlite", "")
                db_map[db_id] = os.path.join(root, file)

    print(f">>> Found {len(db_map)} SQLite databases in {data_dir}.")
    print(f">>> Total dev set samples to evaluate: {len(samples)}")

    passed_semantic_gate = 0
    correct_execution_count = 0
    total_repaired_count = 0

    print(f"\n=================== RUNNING HYBRID SQLGUARD WITH CODES-7B ({len(samples)} SAMPLES, {len(llm_manager.keys)} K2 KEYS) ===================")

    # 4. Start background monitor thread
    stop_monitor_event = threading.Event()
    monitor_thread = threading.Thread(
        target=live_progress_logger,
        args=(stop_monitor_event, len(samples), 15.0),
        daemon=True
    )
    monitor_thread.start()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_sample, sample, db_map) for sample in samples]

        for idx, future in enumerate(tqdm(as_completed(futures), total=len(samples)), 1):
            try:
                final_state = future.result()
                if final_state["validation_passed"]:
                    passed_semantic_gate += 1
                    if final_state["attempt_count"] > 0:
                        total_repaired_count += 1
                if final_state["ex_passed"]:
                    correct_execution_count += 1
            except Exception as e:
                print(f"Sample processing error: {e}")

            if idx % 25 == 0:
                sync_results_to_drive()

    # 5. Stop monitor & perform final sync
    stop_monitor_event.set()
    monitor_thread.join(timeout=1.0)
    sync_results_to_drive()

    total = len(samples)
    print("\n=================== FULL BIRD FINAL BENCHMARK METRICS (7B) ===================")
    print(f"Total Samples Evaluated        : {total}")
    print(f"Passed Semantic Contract Gate  : {passed_semantic_gate}/{total} ({passed_semantic_gate/total*100:.1f}%)")
    print(f"Successfully Repaired Queries  : {total_repaired_count}")
    print(f"BIRD Execution Accuracy (EX)   : {correct_execution_count}/{total} ({correct_execution_count/total*100:.1f}%)")
    print(f"Local Results Directory        : {LOCAL_RESULTS_DIR}")
    print(f"Google Drive Results Directory : {get_drive_results_dir()}")


if __name__ == "__main__":
    try:
        run_full_bird_benchmark(
            limit_samples=None,  # Evaluates all 1,534 samples
            dataset_file="dev.json",
            data_dir=LOCAL_FULL_DEV_DIR,
            max_workers=6
        )
    finally:
        # 1. Ensure any remaining files are synced to Google Drive
        sync_results_to_drive()

        # 2. Disconnect and release the Colab runtime automatically
        print("\n>>> [COMPLETE] Benchmark finished. Shutting down Colab runtime to save compute units...")
        time.sleep(5)  # Short buffer to ensure disk flush

        from google.colab import runtime
        runtime.unassign()


In [ ]:
import os
import re
import json
import time
import shutil
import sqlite3
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Dict, Any, TypedDict, Set, Tuple
from pydantic import BaseModel, Field
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sqlglot
from sqlglot import parse_one, exp
from openai import OpenAI
from tqdm import tqdm
from langgraph.graph import StateGraph, START, END

# =====================================================================
# 0. DIRECTORY PATHS, FULL_DEV RESOLUTION & RESULTS_7B CLEANING
# =====================================================================
DRIVE_BASE_DIR = "/content/drive/MyDrive/SQLGuard_BIRD"
DRIVE_FULL_DEV_CANDIDATES = [
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev", "dev_20240627"),
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev"),
    os.path.join(DRIVE_BASE_DIR, "bird_data")
]

LOCAL_WORKING_DIR = "/content/sqlguard_run"
LOCAL_FULL_DEV_DIR = os.path.join(LOCAL_WORKING_DIR, "full_dev")
LOCAL_RESULTS_DIR = os.path.join(LOCAL_FULL_DEV_DIR, "results_7b_with_evidence_with_column_pruning")
LOCAL_RESULTS_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_results_7b_with_evidence_with_column_pruning.jsonl")
LOCAL_MEMORY_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_failure_memory_7b_with_evidence_with_column_pruning.json")

os.makedirs(LOCAL_WORKING_DIR, exist_ok=True)
file_lock = threading.Lock()
cache_lock = threading.Lock()
gpu_model_lock = threading.Lock()


def resolve_drive_source_dir() -> str:
    """Locates the directory containing dev.json and dev_databases."""
    for candidate in DRIVE_FULL_DEV_CANDIDATES:
        if os.path.exists(candidate):
            has_json = any(os.path.exists(os.path.join(candidate, f)) for f in ["dev.json", "dev_20240627/dev.json"])
            if has_json:
                return candidate
    return DRIVE_FULL_DEV_CANDIDATES[1]


def get_drive_results_dir() -> str:
    """Returns the target Drive results directory inside full_dev/results_7b_with_evidence/."""
    source_dir = resolve_drive_source_dir()
    return os.path.join(source_dir, "results_7b_with_evidence_with_column_pruning")


def clean_and_setup_results_dir():
    """Wipes any previous execution artifacts from full_dev/results_7b_with_evidence both locally and on Drive."""
    # 1. Clean local NVMe results directory
    if os.path.exists(LOCAL_RESULTS_DIR):
        print(f">>> Removing previous local 7B results in {LOCAL_RESULTS_DIR}...")
        shutil.rmtree(LOCAL_RESULTS_DIR)
    os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

    # 2. Clean Google Drive results directory
    drive_results_dir = get_drive_results_dir()
    if os.path.exists(drive_results_dir):
        print(f">>> Removing previous Drive 7B results in {drive_results_dir}...")
        shutil.rmtree(drive_results_dir)
    os.makedirs(drive_results_dir, exist_ok=True)

    print(">>> [READY] Fresh full_dev/results_7b_with_evidence directory initialized.")


def setup_local_colab_environment():
    """Copies dataset files from Google Drive to local NVMe storage."""
    source_dir = resolve_drive_source_dir()
    if os.path.exists(source_dir) and not os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Staging full_dev dataset from {source_dir} to local NVMe ({LOCAL_FULL_DEV_DIR})...")
        shutil.copytree(source_dir, LOCAL_FULL_DEV_DIR)
        print(">>> Staging complete!")
    elif os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Local full_dev dataset ready at {LOCAL_FULL_DEV_DIR}.")


def sync_results_to_drive():
    """Atomically syncs incremental results and failure memory back to full_dev/results_7b_with_evidence on Drive."""
    try:
        drive_results_dir = get_drive_results_dir()
        os.makedirs(drive_results_dir, exist_ok=True)
        if os.path.exists(LOCAL_RESULTS_FILE):
            shutil.copy(LOCAL_RESULTS_FILE, os.path.join(drive_results_dir, "sqlguard_results_7b_with_evidence_with_column_pruning.jsonl"))
        if os.path.exists(LOCAL_MEMORY_FILE):
            shutil.copy(LOCAL_MEMORY_FILE, os.path.join(drive_results_dir, "sqlguard_failure_memory_7b_with_evidence_with_column_pruning.json"))
        print("\n>>> [SYNC SUCCESS] Checkpointed results to Drive: full_dev/results_7b_with_evidence_with_column_pruning/")
    except Exception as e:
        print(f"\n>>> [SYNC ERROR] Drive backup failed: {e}")


# =====================================================================
# 1. K2-THINK-V2 ROTATING API MANAGER
# =====================================================================
K2_BASE_URL = os.getenv("K2_BASE_URL", "https://api.k2think.ai/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "MBZUAI-IFM/K2-Think-v2")

API_KEYS = [
    "IFM-93mLmFQS2bfYZZIY",
    "IFM-SQJYZjtO76Kk86de",
    "IFM-BDZ81VHhqLyYn6Ka"
]


def clean_reasoning_output(raw_text: Optional[str]) -> str:
    """Strips <think> tags, markdown fences, and extracts clean JSON/SQL."""
    if not raw_text:
        return ""
    cleaned = re.sub(r"<think>.*?</think>", "", str(raw_text), flags=re.DOTALL).strip()

    if "```json" in cleaned:
        cleaned = cleaned.split("```json")[1].split("```")[0].strip()
    elif "```sql" in cleaned:
        cleaned = cleaned.split("```sql")[1].split("```")[0].strip()
    elif "```" in cleaned:
        cleaned = cleaned.split("```")[1].split("```")[0].strip()

    if not cleaned.startswith("{") and "select" in cleaned.lower():
        select_pos = cleaned.lower().find("select")
        if select_pos != -1:
            cleaned = cleaned[select_pos:].strip()
            cleaned = cleaned.replace("```", "").strip()

    return cleaned


class K2DynamicKeyManager:
    """Round-robin load balancer across active API keys with latency fallback."""
    def __init__(self, keys: List[str], base_url: str):
        self.keys = [k for k in keys if k and not k.startswith("YOUR_")]
        self.base_url = base_url
        self.clients = {k: OpenAI(base_url=self.base_url, api_key=k, timeout=60.0) for k in self.keys}
        self.index = 0
        self.lock = threading.Lock()

    def get_client_and_key(self) -> tuple[OpenAI, str]:
        with self.lock:
            key = self.keys[self.index % len(self.keys)]
            self.index += 1
            return self.clients[key], key

    def rotate_away_from(self, slow_key: str):
        with self.lock:
            if self.keys[self.index % len(self.keys)] == slow_key:
                self.index += 1

    def execute_chat_completion(self, prompt: str, max_retries: int = 2) -> str:
        for attempt in range(max_retries + 1):
            client, active_key = self.get_client_and_key()
            start_time = time.time()
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0
                )
                if time.time() - start_time > 50.0:
                    self.rotate_away_from(active_key)
                content = resp.choices[0].message.content if resp.choices else ""
                return clean_reasoning_output(content)
            except Exception:
                self.rotate_away_from(active_key)
                if attempt == max_retries:
                    return ""
                time.sleep(1.0)
        return ""


llm_manager = K2DynamicKeyManager(API_KEYS, K2_BASE_URL)


# =====================================================================
# 2. LOCAL CODES-7B ENGINE (WITH EVIDENCE CHECKPOINT & SFT FORMAT)
# =====================================================================
class LocalCodeSEngine:
    """Thread-safe GPU inference manager for CodeS-7B with evidence conditioning."""
    def __init__(self, model_id: str = "seeklhy/codes-7b-bird-with-evidence"):
        print(f">>> Loading local SQL generator: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        print(">>> CodeS-7B (With Evidence) loaded successfully onto GPU.")

    def generate_sql(self, schema_str: str, question: str, evidence: str) -> str:
        """Constructs the canonical CodeS SFT prompt format to preserve native accuracy."""
        prompt = (
            f"Given the database schema, you need to translate the natural language question into SQL query.\n\n"
            f"[Database schema]\n{schema_str}\n\n"
            f"[Question]\n{question}\n\n"
            f"[Evidence]\n{evidence}\n\n"
            f"[SQL]\nSELECT"
        )
        with gpu_model_lock:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=160,
                    pad_token_id=self.tokenizer.eos_token_id,
                    do_sample=False,
                    num_beams=4
                )
            generated = "SELECT" + self.tokenizer.decode(
                output_ids[0][inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )
        return clean_reasoning_output(generated)


codes_engine = LocalCodeSEngine()


# =====================================================================
# 3. THREAD-SAFE PERSISTENT FAILURE MEMORY
# =====================================================================
class PersistentFailureMemory:
    """Maintains an on-disk few-shot repository of repaired SQL patterns inside full_dev/results_7b_with_evidence/."""
    def __init__(self, memory_filepath: str = LOCAL_MEMORY_FILE):
        self.filepath = memory_filepath
        self.memory: List[Dict[str, Any]] = []
        self.reload()

    def reload(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r") as f:
                    self.memory = json.load(f)
            except Exception:
                self.memory = []
        else:
            self.memory = []

    def record_repair(self, question: str, failed_sql: str, error_feedback: str, fixed_sql: str, error_types: List[str]):
        entry = {
            "question": question,
            "failed_sql": failed_sql,
            "error_feedback": error_feedback,
            "fixed_sql": fixed_sql,
            "error_types": error_types
        }
        with file_lock:
            self.memory.append(entry)
            with open(self.filepath, "w") as f:
                json.dump(self.memory, f, indent=2)

    def retrieve_similar_repairs(self, current_errors: List[str], max_examples: int = 2) -> str:
        with file_lock:
            if not self.memory:
                return ""
            mem_snapshot = list(self.memory)

        retrieved = []
        for entry in reversed(mem_snapshot):
            for err in current_errors:
                if any(err_type.lower() in err.lower() for err_type in entry.get("error_types", [])):
                    retrieved.append(entry)
                    break
            if len(retrieved) >= max_examples:
                break

        if not retrieved:
            retrieved = mem_snapshot[-max_examples:]

        formatted_cases = []
        for idx, item in enumerate(retrieved, 1):
            formatted_cases.append(
                f"[Past Repair Example #{idx}]\n"
                f"Question: {item['question']}\n"
                f"Failed Query: {item['failed_sql']}\n"
                f"Errors: {item['error_feedback']}\n"
                f"Corrected SQL: {item['fixed_sql']}"
            )
        return "\n\n".join(formatted_cases)


global_memory = PersistentFailureMemory()


# =====================================================================
# 4. 7-DIMENSIONAL SEMANTIC CONTRACT SCHEMA (Γ)
# =====================================================================
class TargetProjection(BaseModel):
    entity: str = Field(description="Summary of target projection")
    output_columns: List[str] = Field(default_factory=list, description="Projected expressions")
    granularity: Optional[str] = Field(default=None, description="Granularity level")

class SchemaLinks(BaseModel):
    required_tables: List[str] = Field(default_factory=list, description="List of required tables")
    required_columns: List[str] = Field(default_factory=list, description="Specific columns required for projection, filters, aggregations, or joins (e.g. ['table.col', 'col'])")
    join_keys: List[str] = Field(default_factory=list, description="Join paths")

class Analytics(BaseModel):
    aggregations: List[str] = Field(default_factory=list, description="COUNT, AVG, SUM, MIN, MAX")
    group_by: List[str] = Field(default_factory=list, description="Grouping columns or date slice expressions")
    having: List[str] = Field(default_factory=list, description="HAVING conditions")

class RankingCardinality(BaseModel):
    order_by: List[str] = Field(default_factory=list, description="Sort expressions")
    direction: Optional[str] = Field(default=None, description="ASC or DESC")
    limit: Optional[int] = Field(default=None, description="LIMIT top-k cap")

class SemanticContract(BaseModel):
    target_projection: TargetProjection
    schema_links: SchemaLinks
    predicates: List[str] = Field(default_factory=list, description="WHERE filters preserving INTEGER affinity")
    analytics: Analytics
    ranking_cardinality: RankingCardinality
    read_only: bool = Field(default=True, description="Strict read-only safety flag")
    ambiguity_flag: bool = Field(default=False, description="Ambiguity status")


# =====================================================================
# 5. ROBUST STRUCTURED SCHEMA EXTRACTOR & PK/FK-PRESERVING PRUNER
# =====================================================================
SCHEMA_DICT_CACHE: Dict[str, Dict[str, Any]] = {}


def extract_structured_schema(db_path: Optional[str]) -> Dict[str, Any]:
    """Extracts tables, typed columns, PKs, FKs, and representative sample values."""
    if not db_path or not os.path.exists(db_path):
        return {"tables": {}, "foreign_keys": []}

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%';")
        table_names = [r[0] for r in cursor.fetchall()]

        schema_dict: Dict[str, Any] = {"tables": {}, "foreign_keys": []}

        for t in table_names:
            cursor.execute(f"PRAGMA table_info('{t}');")
            cols_info = cursor.fetchall()  # (cid, name, type, notnull, dflt_value, pk)

            col_list: List[Tuple[str, str]] = []
            pk_cols: Set[str] = set()

            for c in cols_info:
                col_name = str(c[1])
                col_type = str(c[2]).upper() if c[2] else 'TEXT'
                is_pk = c[5] > 0
                col_list.append((col_name, col_type))
                if is_pk:
                    pk_cols.add(col_name.lower())

            # Distinct sample values for text/categorical columns
            samples: Dict[str, List[Any]] = {}
            sampled_count = 0
            for c in cols_info:
                if sampled_count >= 3:
                    break
                col_name = c[1]
                try:
                    cursor.execute(f"SELECT DISTINCT \"{col_name}\" FROM \"{t}\" WHERE \"{col_name}\" IS NOT NULL LIMIT 3;")
                    vals = [r[0] for r in cursor.fetchall() if r[0] is not None]
                    if vals:
                        samples[col_name] = vals
                        sampled_count += 1
                except Exception:
                    continue

            # Foreign key extraction: (id, seq, parent_table, from_col, to_col, ...)
            cursor.execute(f"PRAGMA foreign_key_list('{t}');")
            fks = cursor.fetchall()
            for fk in fks:
                from_col = fk[3]
                to_table = fk[2]
                to_col = fk[4]
                if from_col and to_table and to_col:
                    schema_dict["foreign_keys"].append({
                        "from_table": t,
                        "from_col": from_col,
                        "to_table": to_table,
                        "to_col": to_col,
                        "raw": f"{t}.{from_col} = {to_table}.{to_col}"
                    })

            schema_dict["tables"][t] = {
                "columns": col_list,
                "primary_keys": pk_cols,
                "samples": samples
            }

        conn.close()
        return schema_dict
    except Exception as e:
        return {"tables": {}, "foreign_keys": []}


def get_cached_structured_schema(db_id: str, db_path: Optional[str]) -> Dict[str, Any]:
    with cache_lock:
        if db_id in SCHEMA_DICT_CACHE:
            return SCHEMA_DICT_CACHE[db_id]

    data = extract_structured_schema(db_path)
    with cache_lock:
        SCHEMA_DICT_CACHE[db_id] = data
    return data


def format_pruned_schema_for_codes(
    schema_dict: Dict[str, Any],
    required_tables: List[str],
    required_columns: List[str]
) -> str:
    """
    Constructs CodeS canonical [Database schema] string with robust PK/FK preservation.
    Never starves join paths while pruning distractor columns.
    """
    tables_dict = schema_dict.get("tables", {})
    all_table_names = list(tables_dict.keys())
    if not all_table_names:
        return "Schema unavailable."

    # 1. Resolve Target Tables
    clean_req_tables = {t.lower().strip("`'\"[] ") for t in required_tables if t}
    target_tables = [t for t in all_table_names if t.lower().strip("`'\"[] ") in clean_req_tables]

    # Fallback: keep all tables if Intent Agent returned empty or non-matching tables
    if not target_tables:
        target_tables = all_table_names

    target_tables_lower = {t.lower() for t in target_tables}

    # 2. Extract Essential Foreign Keys & Associated Join Columns
    fks = schema_dict.get("foreign_keys", [])
    essential_fk_cols: Set[str] = set()       # Qualified 'table.col'
    essential_bare_fk_cols: Set[str] = set()  # Bare 'col'
    relevant_fk_strings: List[str] = []

    for fk in fks:
        from_t = fk["from_table"].lower()
        to_t = fk["to_table"].lower()
        from_c = fk["from_col"].lower()
        to_c = fk["to_col"].lower()

        # If both tables are in target set, or at least one is in target set
        if from_t in target_tables_lower or to_t in target_tables_lower:
            relevant_fk_strings.append(fk["raw"])
            essential_fk_cols.add(f"{from_t}.{from_c}")
            essential_fk_cols.add(f"{to_t}.{to_c}")
            essential_bare_fk_cols.add(from_c)
            essential_bare_fk_cols.add(to_c)

    # 3. Parse & Normalize User/LLM Required Columns
    clean_target_cols: Set[str] = set()
    clean_bare_target_cols: Set[str] = set()

    for c in required_columns:
        if not c:
            continue
        cleaned = c.lower().strip("`'\"[] ")
        clean_target_cols.add(cleaned)
        if "." in cleaned:
            clean_bare_target_cols.add(cleaned.split(".")[-1].strip("`'\"[] "))
        else:
            clean_bare_target_cols.add(cleaned)

    # 4. Construct Pruned Table Blocks
    lines = []
    for t in target_tables:
        t_info = tables_dict[t]
        cols: List[Tuple[str, str]] = t_info["columns"]
        pks: Set[str] = t_info.get("primary_keys", set())
        samples: Dict[str, List[Any]] = t_info.get("samples", {})

        col_strs = []
        for col_name, col_type in cols:
            col_lower = col_name.lower()
            qualified_col = f"{t.lower()}.{col_lower}"

            # Retention Criteria:
            # 1. Primary Key (PK) - NEVER drop
            # 2. Foreign Key (FK) - NEVER drop
            # 3. Explicitly requested in required_columns (qualified or bare)
            # 4. Fallback if no target columns were extracted at all
            is_pk = col_lower in pks
            is_fk = (qualified_col in essential_fk_cols) or (col_lower in essential_bare_fk_cols)
            is_explicit = (qualified_col in clean_target_cols) or (col_lower in clean_bare_target_cols)
            has_no_column_filter = (len(clean_target_cols) == 0)

            if is_pk or is_fk or is_explicit or has_no_column_filter:
                val_examples = samples.get(col_name, [])
                if val_examples:
                    col_strs.append(f"({col_name}, {col_type}. Value examples: {val_examples})")
                else:
                    col_strs.append(f"({col_name}, {col_type})")

        # Safety Guarantee: If pruning resulted in 0 columns for this table, retain all columns
        if not col_strs:
            for col_name, col_type in cols:
                val_examples = samples.get(col_name, [])
                if val_examples:
                    col_strs.append(f"({col_name}, {col_type}. Value examples: {val_examples})")
                else:
                    col_strs.append(f"({col_name}, {col_type})")

        lines.append(f"# Table: {t}\n[\n  " + ",\n  ".join(col_strs) + "\n]")

    # 5. Append Relevant Foreign Keys
    if relevant_fk_strings:
        lines.append("[Foreign keys]\n" + "\n".join(relevant_fk_strings))

    return "\n".join(lines)


# =====================================================================
# 6. DUAL-GATE VALIDATOR (AST ASSERTIONS + SQLITE DRY-RUN CHECK)
# =====================================================================
class SQLGuardValidator:
    """Validates structural semantics with AST rules and checks engine executable validity."""
    def validate(self, sql: str, contract: SemanticContract, db_path: Optional[str]) -> Dict[str, Any]:
        errors = []
        error_types = []

        if not sql or not sql.strip():
            return {"passed": False, "errors": ["Generated SQL was empty."], "error_types": ["Syntax"]}

        # 1. AST Parse Check
        try:
            parsed = parse_one(sql, read="sqlite")
        except Exception as e:
            return {"passed": False, "errors": [f"AST Parse Error: {str(e)}"], "error_types": ["Syntax"]}

        if not isinstance(parsed, exp.Select):
            return {
                "passed": False,
                "errors": ["Unit Test [V_safety] Failed: Non-SELECT operation blocked."],
                "error_types": ["Safety"]
            }

        query_tables = {t.name.lower().strip("`'\"[] ") for t in parsed.find_all(exp.Table)}
        query_columns_bare = {c.name.lower().strip("`'\"[] ") for c in parsed.find_all(exp.Column)}

        # 2. Table Verification
        for req_t in contract.schema_links.required_tables:
            req_t_clean = req_t.lower().strip("`'\"[] ")
            if req_t_clean and req_t_clean not in query_tables:
                errors.append(f"Unit Test [Schema Table] Failed: Required table '{req_t}' missing.")
                error_types.append("SchemaTable")

        # 3. Column Verification (Bare vs Qualified)
        for req_c in contract.schema_links.required_columns:
            req_clean = req_c.lower().strip("`'\"[] ")
            req_bare = req_clean.split(".")[-1].strip("`'\"[] ")
            if req_bare and req_bare not in query_columns_bare:
                errors.append(f"Unit Test [Schema Column] Failed: Required column '{req_c}' missing.")
                error_types.append("SchemaColumn")

        # 4. Aggregations
        if contract.analytics.aggregations:
            ast_funcs = set()
            if parsed.find(exp.Count): ast_funcs.add("count")
            if parsed.find(exp.Sum): ast_funcs.add("sum")
            if parsed.find(exp.Avg): ast_funcs.add("avg")
            if parsed.find(exp.Max): ast_funcs.add("max")
            if parsed.find(exp.Min): ast_funcs.add("min")

            for req_agg in contract.analytics.aggregations:
                req_clean = req_agg.lower().strip()
                matched = any(kw in req_clean and kw in ast_funcs for kw in ["count", "sum", "avg", "max", "min"])
                if not matched:
                    errors.append(f"Unit Test [Aggregation] Failed: Missing required function '{req_agg}'.")
                    error_types.append("Aggregation")

        # 5. Predicates
        if contract.predicates:
            has_where = parsed.find(exp.Where) is not None
            has_having = parsed.find(exp.Having) is not None
            if not (has_where or has_having):
                errors.append("Unit Test [Predicate] Failed: Filters specified in contract but WHERE/HAVING missing.")
                error_types.append("Predicate")

        # 6. Group By
        if contract.analytics.group_by and not parsed.find(exp.Group):
            errors.append("Unit Test [GroupBy] Failed: Contract specifies grouping but GROUP BY clause missing.")
            error_types.append("GroupBy")

        # 7. Order By & Limit
        if contract.ranking_cardinality.direction and not parsed.find(exp.Order):
            errors.append("Unit Test [Ranking] Failed: Contract specifies sort order but ORDER BY clause missing.")
            error_types.append("Ranking")

        if contract.ranking_cardinality.limit is not None and not parsed.find(exp.Limit):
            errors.append("Unit Test [Limit] Failed: Contract specifies top-k cap but LIMIT clause missing.")
            error_types.append("Limit")

        # 8. SQLite Engine Runtime Dry-Run Check
        if len(errors) == 0 and db_path and os.path.exists(db_path):
            try:
                conn = sqlite3.connect(db_path, timeout=5.0)
                cursor = conn.cursor()
                cursor.execute(sql)
                rows = cursor.fetchall()
                conn.close()
                if len(rows) == 0 and not contract.ambiguity_flag:
                    errors.append("Runtime Warning: Query executed successfully but returned 0 rows (Empty Set).")
                    error_types.append("EmptyResult")
            except sqlite3.OperationalError as op_err:
                errors.append(f"SQLite OperationalError: {str(op_err)}")
                error_types.append("RuntimeError")
            except Exception as ex_err:
                errors.append(f"SQLite Execution Exception: {str(ex_err)}")
                error_types.append("RuntimeError")

        return {
            "passed": len(errors) == 0,
            "errors": errors,
            "error_types": list(set(error_types))
        }


# =====================================================================
# 7. LANGGRAPH WORKFLOW NODES
# =====================================================================
class SQLGuardState(TypedDict):
    question: str
    evidence: str
    db_id: str
    db_path: str
    gold_sql: str
    schema_dict: Dict[str, Any]
    pruned_schema_str: str
    contract: Optional[SemanticContract]
    current_sql: str
    validation_passed: bool
    validation_errors: List[str]
    validation_error_types: List[str]
    attempt_count: int
    max_attempts: int
    initial_failed_sql: str
    initial_errors: List[str]
    initial_error_types: List[str]
    ex_passed: bool
    execution_result: Optional[List[Any]]
    audit_record: Dict[str, Any]


def schema_linker_node(state: SQLGuardState) -> Dict[str, Any]:
    schema_dict = get_cached_structured_schema(state["db_id"], state["db_path"])
    return {"schema_dict": schema_dict}


def intent_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    tables_dict = state["schema_dict"].get("tables", {})
    available_tables = list(tables_dict.keys())

    # Build schema summary for the Intent Agent
    schema_summary_lines = []
    for t, info in tables_dict.items():
        cols = [f"{c[0]} ({c[1]})" for c in info["columns"]]
        schema_summary_lines.append(f"TABLE {t}: " + ", ".join(cols))

    schema_summary = "\n".join(schema_summary_lines)

    prompt = f"""You are the Intent Agent for SQLGuard. Extract a 7-dimensional Semantic Contract as a JSON object.

### MANDATORY SCHEMA LINKING & PRUNING INSTRUCTIONS:
1. SCHEMA SELECTION:
   - Identify all `required_tables` from the available tables: {available_tables}.
   - Identify the 4-8 specific `required_columns` needed for SELECT projection, WHERE filters, aggregations, or JOIN conditions. Format as `table.column` or `column`.
2. DOMAIN EVIDENCE GROUNDING: Strictly follow domain evidence. If evidence indicates date slicing (e.g. SUBSTR(Date, 5, 2) for month, SUBSTR(Date, 1, 4) for year), use that exact expression in output_columns and group_by.
3. SQLITE TYPE COMPLIANCE: If column type is INTEGER (e.g. Date 201301), do NOT wrap numeric numbers in single quotes (use `Date BETWEEN 201301 AND 201312`).
4. BENCHMARK GROUNDING:
   - NAME PROJECTION: Project two separate columns `first_name, last_name` (or `forename, surname`). DO NOT concatenate with `|| ' ' ||`.
   - CASE-INSENSITIVE TEXT FILTERS: For string equality in WHERE, use `COLLATE NOCASE` or `LIKE`.
   - NULL-SAFE SUMS: Always provide `ELSE 0` in conditional aggregation (`SUM(CASE WHEN cond THEN val ELSE 0 END)`).

Database Schema Summary:
{schema_summary}

User Question: {state["question"]}
Domain Evidence: {state["evidence"]}

JSON Schema:
{json.dumps(SemanticContract.model_json_schema())}

Output ONLY the raw JSON object."""

    raw_json = llm_manager.execute_chat_completion(prompt)
    try:
        contract = SemanticContract.model_validate_json(raw_json)
    except Exception:
        contract = SemanticContract(
            target_projection=TargetProjection(entity=state["question"]),
            schema_links=SchemaLinks(required_tables=available_tables),
            analytics=Analytics(),
            ranking_cardinality=RankingCardinality()
        )

    # Format CodeS canonical pruned schema with PK/FK preservation
    pruned_schema = format_pruned_schema_for_codes(
        schema_dict=state["schema_dict"],
        required_tables=contract.schema_links.required_tables,
        required_columns=contract.schema_links.required_columns
    )

    return {"contract": contract, "pruned_schema_str": pruned_schema}


def generator_decomposer_node(state: SQLGuardState) -> Dict[str, Any]:
    generated_sql = codes_engine.generate_sql(
        schema_str=state["pruned_schema_str"],
        question=state["question"],
        evidence=state["evidence"]
    )
    return {"current_sql": generated_sql}


def hybrid_validator_node(state: SQLGuardState) -> Dict[str, Any]:
    validator = SQLGuardValidator()
    val = validator.validate(state["current_sql"], state["contract"], state["db_path"])

    updates = {
        "validation_passed": val["passed"],
        "validation_errors": val["errors"],
        "validation_error_types": val.get("error_types", [])
    }

    if not val["passed"] and state["attempt_count"] == 0:
        updates["initial_failed_sql"] = state["current_sql"]
        updates["initial_errors"] = val["errors"]
        updates["initial_error_types"] = val.get("error_types", [])

    return updates


def repair_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    memory_ctx = global_memory.retrieve_similar_repairs(state["validation_errors"])
    contract_json = state["contract"].model_dump_json() if state["contract"] else "{}"

    prompt = f"""Repair the following SQLite query to satisfy the Semantic Contract, SQLite Engine constraints, and Domain Evidence.

Schema:
{state["pruned_schema_str"]}

Question: {state["question"]}
Evidence: {state["evidence"]}
Semantic Contract: {contract_json}

[FAILED CANDIDATE SQL]:
{state["current_sql"]}

[VALIDATION & RUNTIME ERRORS]:
{chr(10).join(state["validation_errors"]) }
"""
    if memory_ctx:
        prompt += f"\n[SIMILAR PAST SUCCESSFUL REPAIRS]:\n{memory_ctx}\n"

    prompt += "\nOutput raw repaired SQL inside a ```sql codeblock."

    repaired_sql = llm_manager.execute_chat_completion(prompt)
    if not repaired_sql:
        repaired_sql = state["current_sql"]

    return {
        "current_sql": repaired_sql,
        "attempt_count": state["attempt_count"] + 1
    }


def execution_gate_node(state: SQLGuardState) -> Dict[str, Any]:
    ex_passed = False
    pred_res = None

    if state["validation_passed"] and state["db_path"] and os.path.exists(state["db_path"]):
        conn = sqlite3.connect(state["db_path"], timeout=10.0)
        cursor = conn.cursor()
        try:
            cursor.execute(state["current_sql"])
            pred_res = cursor.fetchall()

            cursor.execute(state["gold_sql"])
            gold_res = cursor.fetchall()

            # Robust float normalization comparison
            def normalize_cell(val):
                if isinstance(val, float):
                    return round(val, 2)
                return val

            def normalize_rows(rows):
                if not rows:
                    return []
                return [tuple(normalize_cell(c) for c in row) for row in rows]

            pred_norm = normalize_rows(pred_res)
            gold_norm = normalize_rows(gold_res)

            ex_passed = (pred_norm == gold_norm or set(pred_norm) == set(gold_norm))
        except Exception:
            ex_passed = False
        finally:
            conn.close()

    if state["validation_passed"] and state["attempt_count"] > 0 and state["initial_failed_sql"]:
        global_memory.record_repair(
            question=state["question"],
            failed_sql=state["initial_failed_sql"],
            error_feedback="\n".join(state["initial_errors"]),
            fixed_sql=state["current_sql"],
            error_types=state["initial_error_types"]
        )

    audit_record = {
        "question": state["question"],
        "db_id": state["db_id"],
        "contract": state["contract"].model_dump() if state["contract"] else {},
        "final_sql": state["current_sql"],
        "validation_passed": state["validation_passed"],
        "attempt_count": state["attempt_count"],
        "ex_passed": ex_passed
    }

    return {
        "ex_passed": ex_passed,
        "execution_result": pred_res,
        "audit_record": audit_record
    }


# =====================================================================
# 8. LANGGRAPH COMPILATION & LIVE BACKGROUND MONITOR
# =====================================================================
def validation_router(state: SQLGuardState) -> str:
    if state["validation_passed"]:
        return "execution_gate"
    if state["attempt_count"] < state["max_attempts"]:
        return "repair_agent"
    return "execution_gate"


workflow = StateGraph(SQLGuardState)

workflow.add_node("schema_linker", schema_linker_node)
workflow.add_node("intent_agent", intent_agent_node)
workflow.add_node("generator_decomposer", generator_decomposer_node)
workflow.add_node("hybrid_validator", hybrid_validator_node)
workflow.add_node("repair_agent", repair_agent_node)
workflow.add_node("execution_gate", execution_gate_node)

workflow.add_edge(START, "schema_linker")
workflow.add_edge("schema_linker", "intent_agent")
workflow.add_edge("intent_agent", "generator_decomposer")
workflow.add_edge("generator_decomposer", "hybrid_validator")

workflow.add_conditional_edges(
    "hybrid_validator",
    validation_router,
    {
        "execution_gate": "execution_gate",
        "repair_agent": "repair_agent"
    }
)

workflow.add_edge("repair_agent", "hybrid_validator")
workflow.add_edge("execution_gate", END)

sqlguard_app = workflow.compile()


def process_single_sample(sample: Dict[str, Any], db_map: Dict[str, str]) -> Dict[str, Any]:
    db_id = sample["db_id"]
    db_path = db_map.get(db_id, "")

    initial_state: SQLGuardState = {
        "question": sample["question"],
        "evidence": sample.get("evidence", ""),
        "db_id": db_id,
        "db_path": db_path,
        "gold_sql": sample["SQL"],
        "schema_dict": {},
        "pruned_schema_str": "",
        "contract": None,
        "current_sql": "",
        "validation_passed": False,
        "validation_errors": [],
        "validation_error_types": [],
        "attempt_count": 0,
        "max_attempts": 3,
        "initial_failed_sql": "",
        "initial_errors": [],
        "initial_error_types": [],
        "ex_passed": False,
        "execution_result": None,
        "audit_record": {}
    }

    final_state = sqlguard_app.invoke(initial_state)

    with file_lock:
        with open(LOCAL_RESULTS_FILE, "a") as f_out:
            f_out.write(json.dumps(final_state["audit_record"]) + "\n")

    return final_state


def live_progress_logger(stop_event: threading.Event, total_target: int, poll_interval: float = 10.0):
    """Background thread that prints real-time accuracy and recovery metrics for 7B run."""
    while not stop_event.is_set():
        if os.path.exists(LOCAL_RESULTS_FILE):
            records = []
            with file_lock:
                with open(LOCAL_RESULTS_FILE, "r") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            try:
                                records.append(json.loads(line))
                            except json.JSONDecodeError:
                                continue

            n = len(records)
            if n > 0:
                ex_pass = sum(1 for r in records if r.get("ex_passed", False))
                val_pass = sum(1 for r in records if r.get("validation_passed", False))
                repairs = [r for r in records if r.get("attempt_count", 0) > 0]
                rep_ex = sum(1 for r in repairs if r.get("ex_passed", False))

                pct_done = (n / total_target) * 100
                ex_acc = (ex_pass / n) * 100
                val_rate = (val_pass / n) * 100
                rep_acc = (rep_ex / len(repairs) * 100) if repairs else 0.0

                print(
                    f"\n[LIVE 7B MONITOR] Evaluated: {n}/{total_target} ({pct_done:.1f}%) | "
                    f"EX Acc: {ex_acc:.2f}% | AST Valid: {val_rate:.1f}% | "
                    f"Repairs Recovered: {rep_ex}/{len(repairs)} ({rep_acc:.1f}%)"
                )

        stop_event.wait(poll_interval)


def run_full_bird_benchmark(
    limit_samples: Optional[int] = None,
    dataset_file: str = "dev.json",
    data_dir: str = LOCAL_FULL_DEV_DIR,
    max_workers: int = 6
):
    # 1. Setup local environment & clean previous results in results_7b_with_evidence
    setup_local_colab_environment()
    clean_and_setup_results_dir()
    global_memory.reload()

    # 2. Locate evaluation JSON
    json_path = None
    for root, _, files in os.walk(data_dir):
        if dataset_file in files:
            json_path = os.path.join(root, dataset_file)
            break

    if not json_path:
        raise FileNotFoundError(f"Could not locate {dataset_file} in '{data_dir}'.")

    with open(json_path, "r") as f:
        full_data = json.load(f)
        samples = full_data[:limit_samples] if limit_samples is not None else full_data

    # 3. Map SQLite databases
    db_map = {}
    for root, _, files in os.walk(data_dir):
        for file in files:
            if file.endswith(".sqlite"):
                db_id = file.replace(".sqlite", "")
                db_map[db_id] = os.path.join(root, file)

    print(f">>> Found {len(db_map)} SQLite databases in {data_dir}.")
    print(f">>> Total dev set samples to evaluate: {len(samples)}")

    passed_semantic_gate = 0
    correct_execution_count = 0
    total_repaired_count = 0

    print(f"\n=================== RUNNING HYBRID SQLGUARD WITH CODES-7B ({len(samples)} SAMPLES, {len(llm_manager.keys)} K2 KEYS) ===================")

    # 4. Start background monitor thread
    stop_monitor_event = threading.Event()
    monitor_thread = threading.Thread(
        target=live_progress_logger,
        args=(stop_monitor_event, len(samples), 15.0),
        daemon=True
    )
    monitor_thread.start()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_sample, sample, db_map) for sample in samples]

        for idx, future in enumerate(tqdm(as_completed(futures), total=len(samples)), 1):
            try:
                final_state = future.result()
                if final_state["validation_passed"]:
                    passed_semantic_gate += 1
                    if final_state["attempt_count"] > 0:
                        total_repaired_count += 1
                if final_state["ex_passed"]:
                    correct_execution_count += 1
            except Exception as e:
                print(f"Sample processing error: {e}")

            if idx % 25 == 0:
                sync_results_to_drive()

    # 5. Stop monitor & perform final sync
    stop_monitor_event.set()
    monitor_thread.join(timeout=1.0)
    sync_results_to_drive()

    total = len(samples)
    print("\n=================== FULL BIRD FINAL BENCHMARK METRICS (7B) ===================")
    print(f"Total Samples Evaluated        : {total}")
    print(f"Passed Semantic Contract Gate  : {passed_semantic_gate}/{total} ({passed_semantic_gate/total*100:.1f}%) ")
    print(f"Successfully Repaired Queries  : {total_repaired_count}")
    print(f"BIRD Execution Accuracy (EX)   : {correct_execution_count}/{total} ({correct_execution_count/total*100:.1f}%) ")
    print(f"Local Results Directory        : {LOCAL_RESULTS_DIR}")
    print(f"Google Drive Results Directory : {get_drive_results_dir()}")


if __name__ == "__main__":
    try:
        run_full_bird_benchmark(
            limit_samples=None,  # Evaluates all 1,534 samples
            dataset_file="dev.json",
            data_dir=LOCAL_FULL_DEV_DIR,
            max_workers=6
        )
    finally:
        # 1. Ensure any remaining files are synced to Google Drive
        sync_results_to_drive()

        # 2. Disconnect and release the Colab runtime automatically
        print("\n>>> [COMPLETE] Benchmark finished. Shutting down Colab runtime to save compute units...")
        time.sleep(5)  # Short buffer to ensure disk flush

        from google.colab import runtime
        runtime.unassign()


>>> Loading local SQL generator: seeklhy/codes-7b-bird-with-evidence...


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...


[LIVE 7B MONITOR] Evaluated: 94/1534 (6.1%) | EX Acc: 32.98% | AST Valid: 80.9% | Repairs Recovered: 28/84 (33.3%)


  6%|▌         | 95/1534 [11:42<4:30:46, 11.29s/it]


[LIVE 7B MONITOR] Evaluated: 95/1534 (6.2%) | EX Acc: 32.63% | AST Valid: 81.1% | Repairs Recovered: 28/85 (32.9%)

[LIVE 7B MONITOR] Evaluated: 95/1534 (6.2%) | EX Acc: 32.63% | AST Valid: 81.1% | Repairs Recovered: 28/85 (32.9%)


  6%|▋         | 96/1534 [12:05<3:01:05,  7.56s/it]



>>> [SYNC SUCCESS] Checkpointed results to Drive: full_dev/results_7b_with_evidence_with_column_pruning/

>>> [COMPLETE] Benchmark finished. Shutting down Colab runtime to save compute units...

[LIVE 7B MONITOR] Evaluated: 97/1534 (6.3%) | EX Acc: 32.99% | AST Valid: 80.4% | Repairs Recovered: 29/87 (33.3%)


### Download and Stage SQLFixAgent Pre-filtered Data (SIC)
We will download the pre-processed data from the SQLFixAgent repository to Google Drive and then copy it to the local NVMe storage for fast access during inference.

In [ ]:
import os
import shutil

# Reset working directory to avoid 'shell-init' errors
os.chdir('/content')

# Define paths
DRIVE_SQLFIX_DIR = "/content/drive/MyDrive/SQLGuard_BIRD/sqlfix_data"
zip_path = os.path.join(DRIVE_SQLFIX_DIR, "data.zip")
extract_path = os.path.join(DRIVE_SQLFIX_DIR, "extracted")
LOCAL_SIC_DIR = "/content/sqlfix_sic"

os.makedirs(DRIVE_SQLFIX_DIR, exist_ok=True)

# 1. Clean up corrupted or small files
if os.path.exists(zip_path):
    # The actual data.zip is ~2.8GB; 5MB is a safe floor to detect LFS pointers
    if os.path.getsize(zip_path) < 5000000:
        print(f">>> File too small ({os.path.getsize(zip_path)} bytes). Deleting pointer file...")
        os.remove(zip_path)

# 2. Download using curl with specific LFS handling
# We use the github.com/user/repo/releases/download or the direct object link if known,
# but here we'll try the most robust redirection-following flags.
if not os.path.exists(zip_path):
    print(">>> Downloading full SQLFixAgent data.zip (~2.8GB) to Google Drive...")
    # Use -L to follow redirects and -C - to allow resuming if interrupted
    !curl -L --header "Accept: application/vnd.github.v3.raw" -o {zip_path} "https://github.com/yilun-zhao/SQLFixAgent/raw/main/data.zip"

# 3. Final Verification and Extraction
if os.path.exists(zip_path) and os.path.getsize(zip_path) > 5000000:
    print(f">>> Success! File size: {os.path.getsize(zip_path) / 1e6:.2f} MB. Extracting...")
    if os.path.exists(extract_path):
        shutil.rmtree(extract_path)
    os.makedirs(extract_path, exist_ok=True)
    !unzip -o -q {zip_path} -d {extract_path}

    # 4. Stage to Local Runtime for fast I/O
    if os.path.exists(LOCAL_SIC_DIR):
        shutil.rmtree(LOCAL_SIC_DIR)

    if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
        print(f">>> Staging SIC data to local runtime: {LOCAL_SIC_DIR}...")
        shutil.copytree(extract_path, LOCAL_SIC_DIR)
        print(">>> Staging complete! You can now use dev_filtered.json from /content/sqlfix_sic")
    else:
        print(">>> Error: Extraction resulted in an empty folder.")
else:
    print(">>> Error: Download still failing to retrieve the full archive.")
    print(">>> TIP: If this persists, manually download data.zip from GitHub and upload it to your Drive at: " + zip_path)

>>> File too small (304241 bytes). Deleting pointer file...
>>> Downloading full SQLFixAgent data.zip (~2.8GB) to Google Drive...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  297k    0  297k    0     0  1180k      0 --:--:-- --:--:-- --:--:-- 1183k
>>> Error: Download still failing to retrieve the full archive.
>>> TIP: If this persists, manually download data.zip from GitHub and upload it to your Drive at: /content/drive/MyDrive/SQLGuard_BIRD/sqlfix_data/data.zip


In [ ]:
!ls -R /content/sqlfix_sic | head -n 20

/content/sqlfix_sic:


In [ ]:
ls

drive/  sample_data/  sqlfix_sic/


In [ ]:
cd sqlfix_sic/

[Errno 2] No such file or directory: 'sqlfix_sic/'
/content


In [ ]:
ls


dev_databases/        results/
dev.json              results_7b/
dev.sql               results_7b_with_evidence/
dev_tables.json       results_7b_with_evidence_with_column_pruning/
dev_tied_append.json


In [ ]:
cd ..

/content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev


In [ ]:
ls


dev_20240627/


In [ ]:
cd ..

/content/drive/MyDrive/SQLGuard_BIRD/bird_data


In [ ]:
ls

full_dev/  minidev/


In [ ]:
cd ..

/content/drive/MyDrive/SQLGuard_BIRD


In [ ]:
ls


bird_data/    sqlguard_evaluation_dashboard.png  sqlguard_results.jsonl
sqlfix_data/  sqlguard_failure_memory.json


In [ ]:
cd sqlfix_data/

/content/drive/MyDrive/SQLGuard_BIRD/sqlfix_data


In [ ]:
pwd

'/content/drive/MyDrive/SQLGuard_BIRD/sqlfix_data'

In [ ]:
ls

sic_ckpts/


In [ ]:
cd sic_ckpts/

/content/drive/MyDrive/SQLGuard_BIRD/sqlfix_data/sic_ckpts


In [ ]:
ls

sic_bird/  sic_bird_with_evidence/  sic_spider/


In [ ]:
cd sic_bird_with_evidence/

/content/drive/MyDrive/SQLGuard_BIRD/sqlfix_data/sic_ckpts/sic_bird_with_evidence


In [ ]:
ls

config.json          merges.txt               tokenizer_config.json  vocab.json
dense_classifier.pt  special_tokens_map.json  tokenizer.json


In [ ]:
pwd

'/content/drive/MyDrive/SQLGuard_BIRD/sqlfix_data/sic_ckpts/sic_bird_with_evidence'

In [ ]:
import os
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel, AutoTokenizer
from typing import Dict, Any, List, Tuple, Set


class SchemaItemClassifierModel(nn.Module):
    """Cross-encoder architecture matching the RESDSQL / CodeS SIC checkpoint."""
    def __init__(self, model_dir: str):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_dir)
        self.encoder = AutoModel.from_pretrained(model_dir, config=self.config)
        self.dropout = nn.Dropout(0.1)
        hidden_size = self.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] token representation
        cls_rep = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_rep)).squeeze(-1)
        return torch.sigmoid(logits)


class SICPruner:
    """Thread-safe schema filter that ranks tables and columns using dense_classifier.pt."""
    def __init__(self, checkpoint_dir: str, device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        print(f">>> Loading Schema Item Classifier from {checkpoint_dir}...")
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)

        self.model = SchemaItemClassifierModel(checkpoint_dir)

        # Load weights from dense_classifier.pt (or pytorch_model.bin)
        weights_path = os.path.join(checkpoint_dir, "dense_classifier.pt")
        if not os.path.exists(weights_path):
            weights_path = os.path.join(checkpoint_dir, "pytorch_model.bin")

        state_dict = torch.load(weights_path, map_location=self.device)

        # Handle state_dict key prefix mismatches if saved wrapped
        if "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        cleaned_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

        try:
            self.model.load_state_dict(cleaned_state_dict, strict=False)
        except Exception:
            # Fallback for classification head only vs full model
            self.model.encoder.load_state_dict(cleaned_state_dict, strict=False)

        self.model.to(self.device)
        self.model.eval()
        print(">>> SIC model loaded successfully.")

    @torch.no_grad()
    def rank_items(self, question: str, items: List[str], batch_size: int = 32) -> List[Tuple[str, float]]:
        """Scores relevance of a list of schema candidate strings against the user question."""
        scores = []
        for i in range(0, len(items), batch_size):
            batch_items = items[i:i + batch_size]
            encoded = self.tokenizer(
                [question] * len(batch_items),
                batch_items,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(self.device)

            probs = self.model(encoded["input_ids"], encoded["attention_mask"]).cpu().tolist()
            if isinstance(probs, float):
                probs = [probs]
            for item, prob in zip(batch_items, probs):
                scores.append((item, prob))
        return scores

    def prune_schema_for_codes(
        self,
        question: str,
        schema_dict: Dict[str, Any],
        top_k_tables: int = 5,
        table_prob_thresh: float = 0.20,
        top_k_cols: int = 8,
        col_prob_thresh: float = 0.15
    ) -> str:
        """
        Filters tables & columns using SIC probability scores with primary/foreign key guarantees.
        """
        tables_dict = schema_dict.get("tables", {})
        all_table_names = list(tables_dict.keys())
        if not all_table_names:
            return "Schema unavailable."

        # 1. Rank Tables
        table_candidates = [f"Table: {t}" for t in all_table_names]
        table_scores = self.rank_items(question, table_candidates)

        # Sort by relevance probability
        ranked_tables = sorted(
            [(t, score) for (t_str, score), t in zip(table_scores, all_table_names)],
            key=lambda x: x[1],
            reverse=True
        )

        # Keep top-K OR tables with P > table_prob_thresh
        selected_tables = [t for t, score in ranked_tables if score >= table_prob_thresh]
        if len(selected_tables) < top_k_tables:
            selected_tables = [t for t, _ in ranked_tables[:top_k_tables]]

        selected_tables_set = set(selected_tables)
        selected_tables_lower = {t.lower() for t in selected_tables}

        # 2. Extract Foreign Keys & Essential Join Columns for Selected Tables
        fks = schema_dict.get("foreign_keys", [])
        essential_fk_cols: Set[str] = set()
        essential_bare_fk_cols: Set[str] = set()
        relevant_fk_strings: List[str] = []

        for fk in fks:
            from_t = fk["from_table"].lower() if isinstance(fk, dict) else fk.split(".")[0].lower()
            to_t = fk["to_table"].lower() if isinstance(fk, dict) else fk.split("=")[1].split(".")[0].strip().lower()

            raw_fk = fk["raw"] if isinstance(fk, dict) else fk

            if from_t in selected_tables_lower or to_t in selected_tables_lower:
                relevant_fk_strings.append(raw_fk)
                if isinstance(fk, dict):
                    essential_fk_cols.add(f"{from_t}.{fk['from_col'].lower()}")
                    essential_fk_cols.add(f"{to_t}.{fk['to_col'].lower()}")
                    essential_bare_fk_cols.add(fk["from_col"].lower())
                    essential_bare_fk_cols.add(fk["to_col"].lower())

        # 3. Rank and Filter Columns for Each Selected Table
        lines = []
        for t in selected_tables:
            t_info = tables_dict[t]
            cols: List[Tuple[str, str]] = t_info["columns"]
            pks: Set[str] = t_info.get("primary_keys", set())
            samples: Dict[str, List[Any]] = t_info.get("samples", {})

            # Prepare column candidates for SIC
            col_candidates = [f"Table: {t}, Column: {c_name} ({c_type})" for c_name, c_type in cols]
            col_scores = self.rank_items(question, col_candidates)

            ranked_cols = sorted(
                [(c, score) for (c_str, score), c in zip(col_scores, cols)],
                key=lambda x: x[1],
                reverse=True
            )

            # High-recall column filter
            keep_col_names = {c[0].lower() for c, score in ranked_cols if score >= col_prob_thresh}
            if len(keep_col_names) < top_k_cols:
                keep_col_names.update([c[0].lower() for c, _ in ranked_cols[:top_k_cols]])

            col_strs = []
            for col_name, col_type in cols:
                c_lower = col_name.lower()
                qualified_col = f"{t.lower()}.{c_lower}"

                # Always keep Primary Keys, Foreign Keys, or SIC high-probability columns
                is_pk = c_lower in pks
                is_fk = (qualified_col in essential_fk_cols) or (c_lower in essential_bare_fk_cols)
                is_sic_picked = c_lower in keep_col_names

                if is_pk or is_fk or is_sic_picked:
                    val_examples = samples.get(col_name, [])
                    if val_examples:
                        col_strs.append(f"({col_name}, {col_type}. Value examples: {val_examples})")
                    else:
                        col_strs.append(f"({col_name}, {col_type})")

            # Fallback if all columns got filtered out
            if not col_strs:
                for col_name, col_type in cols:
                    col_strs.append(f"({col_name}, {col_type})")

            lines.append(f"# Table: {t}\n[\n  " + ",\n  ".join(col_strs) + "\n]")

        # 4. Append Foreign Keys
        if relevant_fk_strings:
            lines.append("[Foreign keys]\n" + "\n".join(relevant_fk_strings))

        return "\n".join(lines)

In [ ]:
import os
import re
import json
import time
import shutil
import sqlite3
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Dict, Any, TypedDict
from pydantic import BaseModel, Field
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sqlglot
from sqlglot import parse_one, exp
from openai import OpenAI
from tqdm import tqdm
from langgraph.graph import StateGraph, START, END

# =====================================================================
# 0. DIRECTORY PATHS, FULL_DEV RESOLUTION & RESULTS_7B CLEANING
# =====================================================================
DRIVE_BASE_DIR = "/content/drive/MyDrive/SQLGuard_BIRD"
DRIVE_FULL_DEV_CANDIDATES = [
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev", "dev_20240627"),
    os.path.join(DRIVE_BASE_DIR, "bird_data", "full_dev"),
    os.path.join(DRIVE_BASE_DIR, "bird_data")
]

LOCAL_WORKING_DIR = "/content/sqlguard_run"
LOCAL_FULL_DEV_DIR = os.path.join(LOCAL_WORKING_DIR, "full_dev")
# This LOCAL_RESULTS_DIR is for the main benchmark, not the ablation
LOCAL_RESULTS_DIR = os.path.join(LOCAL_FULL_DEV_DIR, "results_7b_with_evidence")
LOCAL_RESULTS_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_results_7b_with_evidence.jsonl")
LOCAL_MEMORY_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_failure_memory_7b_with_evidence.json")

# Define a separate directory for ablation test results
LOCAL_ABLATION_RESULTS_DIR = os.path.join(LOCAL_FULL_DEV_DIR, "ablation_results_separate")

os.makedirs(LOCAL_WORKING_DIR, exist_ok=True)
file_lock = threading.Lock()
cache_lock = threading.Lock()
gpu_model_lock = threading.Lock()

def resolve_drive_source_dir() -> str:
    """Locates the directory containing dev.json and dev_databases."""
    for candidate in DRIVE_FULL_DEV_CANDIDATES:
        if os.path.exists(candidate):
            has_json = any(os.path.exists(os.path.join(candidate, f)) for f in ["dev.json", "dev_20240627/dev.json"])
            if has_json:
                return candidate
    return DRIVE_FULL_DEV_CANDIDATES[1]


def get_drive_results_dir() -> str:
    """Returns the target Drive results directory inside full_dev/results_7b_with_evidence/."""
    source_dir = resolve_drive_source_dir()
    return os.path.join(source_dir, "results_7b_with_evidence")


def clean_and_setup_results_dir():
    """Wipes any previous execution artifacts from full_dev/results_7b_with_evidence both locally and on Drive."""
    # 1. Clean local NVMe results directory
    if os.path.exists(LOCAL_RESULTS_DIR):
        print(f">>> Removing previous local 7B results in {LOCAL_RESULTS_DIR}...")
        shutil.rmtree(LOCAL_RESULTS_DIR)
    os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

    # 2. Clean Google Drive results directory
    drive_results_dir = get_drive_results_dir()
    if os.path.exists(drive_results_dir):
        print(f">>> Removing previous Drive 7B results in {drive_results_dir}...")
        shutil.rmtree(drive_results_dir)
    os.makedirs(drive_results_dir, exist_ok=True)

    print(">>> [READY] Fresh full_dev/results_7b_with_evidence directory initialized.")


def setup_local_colab_environment():
    """Copies dataset files from Google Drive to local NVMe storage."""
    source_dir = resolve_drive_source_dir()
    if os.path.exists(source_dir) and not os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Staging full_dev dataset from {source_dir} to local NVMe ({LOCAL_FULL_DEV_DIR})...")
        shutil.copytree(source_dir, LOCAL_FULL_DEV_DIR)
        print(">>> Staging complete!")
    elif os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Local full_dev dataset ready at {LOCAL_FULL_DEV_DIR}.")


def sync_results_to_drive():
    """Atomically syncs incremental results and failure memory back to full_dev/results_7b_with_evidence on Drive."""
    try:
        drive_results_dir = get_drive_results_dir()
        os.makedirs(drive_results_dir, exist_ok=True)
        if os.path.exists(LOCAL_RESULTS_FILE):
            shutil.copy(LOCAL_RESULTS_FILE, os.path.join(drive_results_dir, "sqlguard_results_7b_with_evidence.jsonl"))
        if os.path.exists(LOCAL_MEMORY_FILE):
            shutil.copy(LOCAL_MEMORY_FILE, os.path.join(drive_results_dir, "sqlguard_failure_memory_7b_with_evidence.json"))
        print("\n>>> [SYNC SUCCESS] Checkpointed results to Drive: full_dev/results_7b_with_evidence/")
    except Exception as e:
        print(f"\n>>> [SYNC ERROR] Drive backup failed: {e}")


# =====================================================================
# 1. K2-THINK-V2 ROTATING API MANAGER
# =====================================================================
K2_BASE_URL = os.getenv("K2_BASE_URL", "https://api.k2think.ai/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "MBZUAI-IFM/K2-Think-v2")

API_KEYS = [
    "IFM-api-KEY-1",
    "IFM-API-KEY-2",
    "IFM-api-KEY-3"
]

def clean_reasoning_output(raw_text: Optional[str]) -> str:
    """Strips <think> tags, markdown fences, and extracts clean JSON/SQL."""
    if not raw_text:
        return ""
    cleaned = re.sub(r"<think>.*?</think>", "", str(raw_text), flags=re.DOTALL).strip()

    if "```json" in cleaned:
        cleaned = cleaned.split("```json")[1].split("```")[0].strip()
    elif "```sql" in cleaned:
        cleaned = cleaned.split("```sql")[1].split("```")[0].strip()
    elif "```" in cleaned:
        cleaned = cleaned.split("```")[1].split("```")[0].strip()

    if not cleaned.startswith("{") and "select" in cleaned.lower():
        select_pos = cleaned.lower().find("select")
        if select_pos != -1:
            cleaned = cleaned[select_pos:].strip()
            cleaned = cleaned.replace("```", "").strip()

    return cleaned


class K2DynamicKeyManager:
    """Round-robin load balancer across active API keys with latency fallback."""
    def __init__(self, keys: List[str], base_url: str):
        self.keys = [k for k in keys if k and not k.startswith("YOUR_")]
        self.base_url = base_url
        self.clients = {k: OpenAI(base_url=self.base_url, api_key=k, timeout=60.0) for k in self.keys}
        self.index = 0
        self.lock = threading.Lock()

    def get_client_and_key(self) -> tuple[OpenAI, str]:
        with self.lock:
            key = self.keys[self.index % len(self.keys)]
            self.index += 1
            return self.clients[key], key

    def rotate_away_from(self, slow_key: str):
        with self.lock:
            if self.keys[self.index % len(self.keys)] == slow_key:
                self.index += 1

    def execute_chat_completion(self, prompt: str, max_retries: int = 2) -> str:
        for attempt in range(max_retries + 1):
            client, active_key = self.get_client_and_key()
            start_time = time.time()
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0
                )
                if time.time() - start_time > 50.0:
                    self.rotate_away_from(active_key)
                content = resp.choices[0].message.content if resp.choices else ""
                return clean_reasoning_output(content)
            except Exception:
                self.rotate_away_from(active_key)
                if attempt == max_retries:
                    return ""
                time.sleep(1.0)
        return ""


llm_manager: Optional[K2DynamicKeyManager] = None


def get_llm_manager() -> K2DynamicKeyManager:
    global llm_manager
    if llm_manager is None:
        llm_manager = K2DynamicKeyManager(API_KEYS, K2_BASE_URL)
    return llm_manager


# =====================================================================
# 2. LOCAL CODES-7B ENGINE (WITH EVIDENCE CHECKPOINT & SFT FORMAT)
# =====================================================================
class LocalCodeSEngine:
    """Thread-safe GPU inference manager for CodeS-7B with evidence conditioning."""
    def __init__(self, model_id: str = "seeklhy/codes-7b-bird-with-evidence"):
        print(f">>> Loading local SQL generator: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        print(">>> CodeS-7B (With Evidence) loaded successfully onto GPU.")

    def generate_sql(self, schema_str: str, question: str, evidence: str) -> str:
        """Constructs the canonical CodeS SFT prompt format to preserve native accuracy."""
        prompt = (
            f"Given the database schema, you need to translate the natural language question into SQL query.\n\n"
            f"[Database schema]\n{schema_str}\n\n"
            f"[Question]\n{question}\n\n"
            f"[Evidence]\n{evidence}\n\n"
            f"[SQL]\nSELECT"
        )
        with gpu_model_lock:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=160,
                    pad_token_id=self.tokenizer.eos_token_id,
                    do_sample=False,
                    num_beams=4
                )
            generated = "SELECT" + self.tokenizer.decode(
                output_ids[0][inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )
        return clean_reasoning_output(generated)


codes_engine: Optional[LocalCodeSEngine] = None


def get_codes_engine() -> LocalCodeSEngine:
    global codes_engine
    if codes_engine is None:
        codes_engine = LocalCodeSEngine()
    return codes_engine


# =====================================================================
# 3. THREAD-SAFE PERSISTENT FAILURE MEMORY
# =====================================================================
class PersistentFailureMemory:
    """Maintains an on-disk few-shot repository of repaired SQL patterns inside full_dev/results_7b_with_evidence/."""
    def __init__(self, memory_filepath: str = LOCAL_MEMORY_FILE):
        self.filepath = memory_filepath
        self.memory: List[Dict[str, Any]] = []
        self.reload()

    def reload(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r") as f:
                    self.memory = json.load(f)
            except Exception:
                self.memory = []
        else:
            self.memory = []

    def record_repair(self, question: str, failed_sql: str, error_feedback: str, fixed_sql: str, error_types: List[str]):
        entry = {
            "question": question,
            "failed_sql": failed_sql,
            "error_feedback": error_feedback,
            "fixed_sql": fixed_sql,
            "error_types": error_types
        }
        with file_lock:
            self.memory.append(entry)
            with open(self.filepath, "w") as f:
                json.dump(self.memory, f, indent=2)

    def retrieve_similar_repairs(self, current_errors: List[str], max_examples: int = 2) -> str:
        with file_lock:
            if not self.memory:
                return ""
            mem_snapshot = list(self.memory)

        retrieved = []
        for entry in reversed(mem_snapshot):
            for err in current_errors:
                if any(err_type.lower() in err.lower() for err_type in entry.get("error_types", [])):
                    retrieved.append(entry)
                    break
            if len(retrieved) >= max_examples:
                break

        if not retrieved:
            retrieved = mem_snapshot[-max_examples:]

        formatted_cases = []
        for idx, item in enumerate(retrieved, 1):
            formatted_cases.append(
                f"[Past Repair Example #{idx}]\n"
                f"Question: {item['question']}\n"
                f"Failed Query: {item['failed_sql']}\n"
                f"Errors: {item['error_feedback']}\n"
                f"Corrected SQL: {item['fixed_sql']}"
            )
        return "\n\n".join(formatted_cases)


global_memory = PersistentFailureMemory()


# =====================================================================
# 4. 7-DIMENSIONAL SEMANTIC CONTRACT SCHEMA (Γ)
# =====================================================================
class TargetProjection(BaseModel):
    entity: str = Field(description="Summary of target projection")
    output_columns: List[str] = Field(default_factory=list, description="Projected expressions")
    granularity: Optional[str] = Field(default=None, description="Granularity level")

class SchemaLinks(BaseModel):
    required_tables: List[str] = Field(default_factory=list, description="Required tables")
    required_columns: List[str] = Field(default_factory=list, description="Required columns")
    join_keys: List[str] = Field(default_factory=list, description="Join paths")

class Analytics(BaseModel):
    aggregations: List[str] = Field(default_factory=list, description="COUNT, AVG, SUM, MIN, MAX")
    group_by: List[str] = Field(default_factory=list, description="Grouping columns or date slice expressions")
    having: List[str] = Field(default_factory=list, description="HAVING conditions")

class RankingCardinality(BaseModel):
    order_by: List[str] = Field(default_factory=list, description="Sort expressions")
    direction: Optional[str] = Field(default=None, description="ASC or DESC")
    limit: Optional[int] = Field(default=None, description="LIMIT top-k cap")

class SemanticContract(BaseModel):
    target_projection: TargetProjection
    schema_links: SchemaLinks
    predicates: List[str] = Field(default_factory=list, description="WHERE filters preserving INTEGER affinity")
    analytics: Analytics
    ranking_cardinality: RankingCardinality
    read_only: bool = Field(default=True, description="Strict read-only safety flag")
    ambiguity_flag: bool = Field(default=False, description="Ambiguity status")


# =====================================================================
# 5. COMPACT SCHEMA EXTRACTOR & CACHING
# =====================================================================
SCHEMA_CACHE: Dict[str, str] = {}


def extract_compact_schema(db_path: Optional[str]) -> str:
    """Builds a token-efficient, type-annotated SQLite schema description."""
    if not db_path or not os.path.exists(db_path):
        return "Schema unavailable."

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%';")
        tables = [r[0] for r in cursor.fetchall()]

        schema_lines = []
        for table_name in tables:
            cursor.execute(f"PRAGMA table_info('{table_name}');")
            cols = cursor.fetchall()
            col_desc = [f"{c[1]} ({c[2].upper() or 'TEXT'})" for c in cols]
            schema_lines.append(f"TABLE {table_name} (\n  " + ", ".join(col_desc) + "\n)")

            cursor.execute(f"PRAGMA foreign_key_list('{table_name}');")
            for fk in cursor.fetchall():
                schema_lines.append(f"-- FK: {table_name}.{fk[3]} -> {fk[2]}.{fk[4]}")

            samples = []
            sampled_count = 0
            for col in cols:
                if sampled_count >= 4:
                    break
                col_name = col[1]
                cursor.execute(f"SELECT DISTINCT \"{col_name}\" FROM \"{table_name}\" WHERE \"{col_name}\" IS NOT NULL LIMIT 3;")
                vals = [r[0] for r in cursor.fetchall() if r[0] is not None]
                if vals:
                    samples.append(f"{col_name}: {vals}")
                    sampled_count += 1
            if samples:
                schema_lines.append(f"-- [{table_name} Samples]: " + " | ".join(samples))

        conn.close()
        return "\n".join(schema_lines)
    except Exception as e:
        return f"Error reading schema: {e}"


def get_cached_schema(db_id: str, db_path: Optional[str]) -> str:
    with cache_lock:
        if db_id in SCHEMA_CACHE:
            return SCHEMA_CACHE[db_id]

    compact_schema = extract_compact_schema(db_path)
    with cache_lock:
        SCHEMA_CACHE[db_id] = compact_schema
    return compact_schema


# =====================================================================
# 6. NORMALIZED HYBRID AST VALIDATOR (sqlglot)
# =====================================================================
class SQLGuardValidator:
    def validate(self, sql: str, contract: SemanticContract) -> Dict[str, Any]:
        errors = []
        error_types = []

        if not sql or not sql.strip():
            return {"passed": False, "errors": ["Generated SQL was empty."], "error_types": ["Syntax"]}

        try:
            parsed = parse_one(sql, read="sqlite")
        except Exception as e:
            return {"passed": False, "errors": [f"AST Parse Error: {str(e)}"], "error_types": ["Syntax"]}

        if not isinstance(parsed, exp.Select):
            return {
                "passed": False,
                "errors": ["Unit Test [V_safety] Failed: Non-SELECT operation blocked."],
                "error_types": ["Safety"]
            }

        query_tables = {t.name.lower().strip("`'\"[] ") for t in parsed.find_all(exp.Table)}
        query_columns_bare = {c.name.lower().strip("`'\"[] ") for c in parsed.find_all(exp.Column)}

        # 1. Table Verification
        for req_t in contract.schema_links.required_tables:
            req_t_clean = req_t.lower().strip("`'\"[] ")
            if req_t_clean and req_t_clean not in query_tables:
                errors.append(f"Unit Test [Schema Table] Failed: Required table '{req_t}' missing.")
                error_types.append("SchemaTable")

        # 2. Column Verification (Bare vs Qualified)
        for req_c in contract.schema_links.required_columns:
            req_clean = req_c.lower().strip("`'\"[] ")
            req_bare = req_clean.split(".")[-1].strip("`'\"[] ")
            if req_bare and req_bare not in query_columns_bare:
                errors.append(f"Unit Test [Schema Column] Failed: Required column '{req_c}' missing.")
                error_types.append("SchemaColumn")

        # 3. Aggregations
        if contract.analytics.aggregations:
            ast_funcs = set()
            if parsed.find(exp.Count): ast_funcs.add("count")
            if parsed.find(exp.Sum): ast_funcs.add("sum")
            if parsed.find(exp.Avg): ast_funcs.add("avg")
            if parsed.find(exp.Max): ast_funcs.add("max")
            if parsed.find(exp.Min): ast_funcs.add("min")

            for req_agg in contract.analytics.aggregations:
                req_clean = req_agg.lower().strip()
                matched = any(kw in req_clean and kw in ast_funcs for kw in ["count", "sum", "avg", "max", "min"])
                if not matched:
                    errors.append(f"Unit Test [Aggregation] Failed: Missing required function '{req_agg}'.")
                    error_types.append("Aggregation")

        # 4. Predicates
        if contract.predicates:
            has_where = parsed.find(exp.Where) is not None
            has_having = parsed.find(exp.Having) is not None
            if not (has_where or has_having):
                errors.append("Unit Test [Predicate] Failed: Filters specified in contract but WHERE/HAVING missing.")
                error_types.append("Predicate")

        # 5. Group By
        if contract.analytics.group_by and not parsed.find(exp.Group):
            errors.append("Unit Test [GroupBy] Failed: Contract specifies grouping but GROUP BY clause missing.")
            error_types.append("GroupBy")

        # 6. Order By & Limit
        if contract.ranking_cardinality.direction and not parsed.find(exp.Order):
            errors.append("Unit Test [Ranking] Failed: Contract specifies sort order but ORDER BY clause missing.")
            error_types.append("Ranking")

        if contract.ranking_cardinality.limit is not None and not parsed.find(exp.Limit):
            errors.append("Unit Test [Limit] Failed: Contract specifies top-k cap but LIMIT clause missing.")
            error_types.append("Limit")

        return {
            "passed": len(errors) == 0,
            "errors": errors,
            "error_types": list(set(error_types))
        }


# =====================================================================
# 7. LANGGRAPH WORKFLOW NODES
# =====================================================================
class SQLGuardState(TypedDict):
    question: str
    evidence: str
    db_id: str
    db_path: str
    gold_sql: str
    schema_metadata: str
    contract: Optional[SemanticContract]
    current_sql: str
    validation_passed: bool
    validation_errors: List[str]
    validation_error_types: List[str]
    attempt_count: int
    max_attempts: int
    initial_failed_sql: str
    initial_errors: List[str]
    initial_error_types: List[str]
    ex_passed: bool
    execution_result: Optional[List[Any]]
    audit_record: Dict[str, Any]


def schema_linker_node(state: SQLGuardState) -> Dict[str, Any]:
    schema_meta = get_cached_schema(state["db_id"], state["db_path"])
    return {"schema_metadata": schema_meta}


def intent_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    prompt = f"""You are the Intent Agent for SQLGuard. Extract a 7-dimensional Semantic Contract as a JSON object.\n\n### MANDATORY INSTRUCTIONS:\n1. DOMAIN EVIDENCE GROUNDING: Strictly follow domain evidence. If evidence indicates date slicing (e.g. SUBSTR(Date, 5, 2) for month, SUBSTR(Date, 1, 4) for year), use that exact expression in output_columns and group_by.\n2. SQLITE TYPE COMPLIANCE: If column type is INTEGER (e.g. Date 201301), do NOT wrap numeric numbers in single quotes (use `Date BETWEEN 201301 AND 201312`).\n3. PROJECTION PRECISION: Output ONLY the requested attribute or expression in output_columns.\n\n### BENCHMARK EVALUATION GROUNDING RULES:\n1. NAME PROJECTION: When asked for a person's name or full name, project two separate columns `first_name, last_name` (or `forename, surname`). DO NOT concatenate with `|| ' ' ||`.\n2. CASE-INSENSITIVE TEXT FILTERS: For string equality checks in WHERE clauses, use `COLLATE NOCASE` or `LIKE` (e.g. `Segment = 'Discount' COLLATE NOCASE`).\n3. PROJECTION MINIMALISM: Project ONLY the exact attribute requested. Do not include extra tie-breaker columns or IDs in SELECT unless explicitly requested.\n4. NULL-SAFE SUMS: Always provide `ELSE 0` in conditional aggregation (e.g., `SUM(CASE WHEN condition THEN val ELSE 0 END)`).\n\nSchema:\n{state["schema_metadata"]}\n\nUser Question: {state["question"]}\nDomain Evidence: {state["evidence"]}\n\nJSON Schema:\n{json.dumps(SemanticContract.model_json_schema())}\n\nOutput ONLY the raw JSON object."""

    raw_json = get_llm_manager().execute_chat_completion(prompt)
    try:
        contract = SemanticContract.model_validate_json(raw_json)
    except Exception:
        contract = SemanticContract(
            target_projection=TargetProjection(entity=state["question"]),
            schema_links=SchemaLinks(),
            analytics=Analytics(),
            ranking_cardinality=RankingCardinality()
        )
    return {"contract": contract}


def generator_decomposer_node(state: SQLGuardState) -> Dict[str, Any]:
    generated_sql = get_codes_engine().generate_sql(
        schema_str=state["schema_metadata"],
        question=state["question"],
        evidence=state["evidence"]
    )
    return {"current_sql": generated_sql}


def hybrid_validator_node(state: SQLGuardState) -> Dict[str, Any]:
    validator = SQLGuardValidator()
    val = validator.validate(state["current_sql"], state["contract"])

    updates = {
        "validation_passed": val["passed"],
        "validation_errors": val["errors"],
        "validation_error_types": val.get("error_types", [])
    }

    if not val["passed"] and state["attempt_count"] == 0:
        updates["initial_failed_sql"] = state["current_sql"]
        updates["initial_errors"] = val["errors"]
        updates["initial_error_types"] = val.get("error_types", [])

    return updates


def repair_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    memory_ctx = global_memory.retrieve_similar_repairs(state["validation_errors"])
    contract_json = state["contract"].model_dump_json() if state["contract"] else "{}"

    prompt = f"""Repair the following SQLite query to satisfy the Semantic Contract and Domain Evidence.\n\nSchema:\n{state["schema_metadata"]}\n\nQuestion: {state["question"]}\nEvidence: {state["evidence"]}\nSemantic Contract: {contract_json}\n\n[FAILED CANDIDATE SQL]:\n{state["current_sql"]}\n\n[CONTRACT VALIDATION ERRORS]:\n{chr(10).join(state["validation_errors"]) }\n"""
    if memory_ctx:
        prompt += f"\n[SIMILAR PAST SUCCESSFUL REPAIRS]:\n{memory_ctx}\n"

    prompt += "\nOutput raw repaired SQL inside a ```sql codeblock."

    repaired_sql = get_llm_manager().execute_chat_completion(prompt)
    if not repaired_sql:
        repaired_sql = state["current_sql"]

    return {
        "current_sql": repaired_sql,
        "attempt_count": state["attempt_count"] + 1
    }


def execution_gate_node(state: SQLGuardState) -> Dict[str, Any]:
    ex_passed = False
    pred_res = None

    if state["validation_passed"] and state["db_path"] and os.path.exists(state["db_path"]):
        conn = sqlite3.connect(state["db_path"], timeout=10.0)
        cursor = conn.cursor()
        try:
            cursor.execute(state["current_sql"])
            pred_res = cursor.fetchall()

            cursor.execute(state["gold_sql"])
            gold_res = cursor.fetchall()

            # Robust float normalization comparison
            def normalize_cell(val):
                if isinstance(val, float):
                    return round(val, 2)
                return val

            def normalize_rows(rows):
                if not rows:
                    return []
                return [tuple(normalize_cell(c) for c in row) for row in rows]

            pred_norm = normalize_rows(pred_res)
            gold_norm = normalize_rows(gold_res)

            ex_passed = (pred_norm == gold_norm or set(pred_norm) == set(gold_norm))
        except Exception:
            ex_passed = False
        finally:
            conn.close()

    if state["validation_passed"] and state["attempt_count"] > 0 and state["initial_failed_sql"]:
        global_memory.record_repair(
            question=state["question"],
            failed_sql=state["initial_failed_sql"],
            error_feedback="\n".join(state["initial_errors"]),
            fixed_sql=state["current_sql"],
            error_types=state["initial_error_types"]
        )

    audit_record = {
        "question": state["question"],
        "db_id": state["db_id"],
        "contract": state["contract"].model_dump() if state["contract"] else {},
        "final_sql": state["current_sql"],
        "validation_passed": state["validation_passed"],
        "attempt_count": state["attempt_count"],
        "ex_passed": ex_passed
    }

    return {
        "ex_passed": ex_passed,
        "execution_result": pred_res,
        "audit_record": audit_record
    }


# =====================================================================
# 8. LANGGRAPH COMPILATION & LIVE BACKGROUND MONITOR
# =====================================================================
def validation_router(state: SQLGuardState) -> str:
    if state["validation_passed"]:
        return "execution_gate"
    if state["attempt_count"] < state["max_attempts"]:
        return "repair_agent"
    return "execution_gate"


workflow = StateGraph(SQLGuardState)

workflow.add_node("schema_linker", schema_linker_node)
workflow.add_node("intent_agent", intent_agent_node)
workflow.add_node("generator_decomposer", generator_decomposer_node)
workflow.add_node("hybrid_validator", hybrid_validator_node)
workflow.add_node("repair_agent", repair_agent_node)
workflow.add_node("execution_gate", execution_gate_node)

workflow.add_edge(START, "schema_linker")
workflow.add_edge("schema_linker", "intent_agent")
workflow.add_edge("intent_agent", "generator_decomposer")
workflow.add_edge("generator_decomposer", "hybrid_validator")

workflow.add_conditional_edges(
    "hybrid_validator",
    validation_router,
    {
        "execution_gate": "execution_gate",
        "repair_agent": "repair_agent"
    }
)

workflow.add_edge("repair_agent", "hybrid_validator")
workflow.add_edge("execution_gate", END)

sqlguard_app = workflow.compile()


def process_single_sample(sample: Dict[str, Any], db_map: Dict[str, Any]) -> Dict[str, Any]:
    db_id = sample["db_id"]
    db_path = db_map.get(db_id, "")

    initial_state: SQLGuardState = {
        "question": sample["question"],
        "evidence": sample.get("evidence", ""),
        "db_id": db_id,
        "db_path": db_path,
        "gold_sql": sample["SQL"],
        "schema_metadata": "",
        "contract": None,
        "current_sql": "",
        "validation_passed": False,
        "validation_errors": [],
        "validation_error_types": [],
        "attempt_count": 0,
        "max_attempts": 3,
        "initial_failed_sql": "",
        "initial_errors": [],
        "initial_error_types": [],
        "ex_passed": False,
        "execution_result": None,
        "audit_record": {}
    }

    final_state = sqlguard_app.invoke(initial_state)

    with file_lock:
        with open(LOCAL_RESULTS_FILE, "a") as f_out:
            f_out.write(json.dumps(final_state["audit_record"]) + "\n")

    return final_state

def live_progress_logger(stop_event: threading.Event, total_target: int, poll_interval: float = 10.0, results_file: str = LOCAL_RESULTS_FILE):
    """Background thread that prints real-time accuracy and recovery metrics."""
    while not stop_event.is_set():
        if os.path.exists(results_file):
            records = []
            with file_lock:
                with open(results_file, "r") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            try:
                                records.append(json.loads(line))
                            except json.JSONDecodeError:
                                continue

            n = len(records)
            if n > 0:
                ex_pass = sum(1 for r in records if r.get("ex_passed", False))
                val_pass = sum(1 for r in records if r.get("validation_passed", False))
                repairs = [r for r in records if r.get("attempt_count", 0) > 0]
                rep_ex = sum(1 for r in repairs if r.get("ex_passed", False))

                pct_done = (n / total_target) * 100
                ex_acc = (ex_pass / n) * 100
                val_rate = (val_pass / n) * 100
                rep_acc = (rep_ex / len(repairs) * 100) if repairs else 0.0

                print(
                    f"\n[LIVE MONITOR] Evaluated: {n}/{total_target} ({pct_done:.1f}%) | "
                    f"EX Acc: {ex_acc:.2f}% | AST Valid: {val_rate:.1f}% | "
                    f"Repairs Recovered: {rep_ex}/{len(repairs)} ({rep_acc:.1f}%)"
                )

        stop_event.wait(poll_interval)

# =====================================================================
# 9. ABALATION CONFIG & EXP-1 ENTRYPOINT
# =====================================================================
def run_ablation_experiment(experiment_name, processor, results_file,
                             limit_samples=300, dataset_file="dev.json",
                             data_dir=LOCAL_FULL_DEV_DIR, max_workers=6):
    # 1. Setup local environment & clean previous results for ablation
    setup_local_colab_environment()

    os.makedirs(os.path.dirname(results_file), exist_ok=True)
    if os.path.exists(results_file):
        os.remove(results_file)

    json_path = None
    for root, _, files in os.walk(data_dir):
        if dataset_file in files:
            json_path = os.path.join(root, dataset_file)
            break
    if not json_path:
        raise FileNotFoundError(dataset_file)

    with open(json_path, "r") as f:
        data = json.load(f)
    samples = data[:limit_samples] if limit_samples is not None else data

    db_map = {}
    for root, _, files in os.walk(data_dir):
        for file in files:
            if file.endswith(".sqlite"):
                db_map[file[:-7]] = os.path.join(root, file)

    total_samples_to_evaluate = len(samples)
    print(f"\n=================== RUNNING ABLATION EXPERIMENT: {experiment_name} ({total_samples_to_evaluate} SAMPLES) ===================")

    # Start background monitor thread
    stop_monitor_event = threading.Event()
    monitor_thread = threading.Thread(
        target=live_progress_logger,
        args=(stop_monitor_event, total_samples_to_evaluate, 15.0, results_file),
        daemon=True
    )
    monitor_thread.start()

    ex_count = valid_count = repaired = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(processor, s, db_map) for s in samples]
        for future in tqdm(as_completed(futures), total=len(futures), desc=experiment_name):
            try:
                state = future.result()
                valid_count += int(state["validation_passed"])
                ex_count += int(state["ex_passed"])
                repaired += int(state["attempt_count"] > 0)
            except Exception as e:
                print(f"[ERROR] {e}")

    # Stop monitor & perform final sync
    stop_monitor_event.set()
    monitor_thread.join(timeout=1.0)

    total = len(samples)
    summary = {
        "experiment": experiment_name,
        "total_samples": total,
        "ast_valid_pct": 100 * valid_count / total if total else 0,
        "samples_repaired": repaired,
        "ex_count": ex_count,
        "ex_pct": 100 * ex_count / total if total else 0
    }

    summary_file = os.path.join(os.path.dirname(results_file), "summary.json")
    with open(summary_file, "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    return summary


EXP_NAME = "EXP-1_No_Repair"
EXP_OUTPUT_ROOT = os.path.join(LOCAL_ABLATION_RESULTS_DIR, "ablation_results") # Changed this line
EXP_DIR = os.path.join(EXP_OUTPUT_ROOT, EXP_NAME)
EXP_FILE = os.path.join(EXP_DIR, "results.jsonl")


def make_state(sample, db_map, max_attempts):
    db_id = sample["db_id"]
    return {
        "question": sample["question"],
        "evidence": sample.get("evidence", ""),
        "db_id": db_id,
        "db_path": db_map.get(db_id, ""),
        "gold_sql": sample["SQL"],
        "schema_metadata": "",
        "contract": None,
        "current_sql": "",
        "validation_passed": False,
        "validation_errors": [],
        "validation_error_types": [],
        "attempt_count": 0,
        "max_attempts": max_attempts,
        "initial_failed_sql": "",
        "initial_errors": [],
        "initial_error_types": [],
        "ex_passed": False,
        "execution_result": None,
        "audit_record": {}
    }


def process_single_sample_exp1(sample, db_map):
    state = make_state(sample, db_map, max_attempts=0)
    final_state = sqlguard_app.invoke(state)
    with file_lock:
        with open(EXP_FILE, "a") as f:
            f.write(json.dumps(final_state["audit_record"]) + "\n")
    return final_state


def run_exp1(limit_samples: Optional[int] = None, max_workers: int = 6):
    try:
        summary = run_ablation_experiment(EXP_NAME, process_single_sample_exp1,
                                          EXP_FILE, limit_samples=limit_samples,
                                          max_workers=max_workers)
        return summary
    finally:
        # Ensure any remaining files are synced to Google Drive
        # This sync is for the main results directory, not necessarily the ablation ones
        # A separate sync for ablation results might be needed if they are not picked up by the main sync.
        # However, since the ablation results are in a separate path, we need to ensure they are also synced.
        # Let's add a sync specifically for the ablation results directory.
        try:
            drive_ablation_results_dir = os.path.join(resolve_drive_source_dir(), "ablation_results_separate", EXP_NAME)
            os.makedirs(drive_ablation_results_dir, exist_ok=True)
            if os.path.exists(EXP_FILE):
                shutil.copy(EXP_FILE, os.path.join(drive_ablation_results_dir, "results.jsonl"))
            summary_file = os.path.join(os.path.dirname(EXP_FILE), "summary.json")
            if os.path.exists(summary_file):
                shutil.copy(summary_file, os.path.join(drive_ablation_results_dir, "summary.json"))
            print(f"\n>>> [SYNC SUCCESS] Checkpointed ablation results to Drive: {drive_ablation_results_dir}/")
        except Exception as e:
            print(f"\n>>> [SYNC ERROR] Ablation results Drive backup failed: {e}")

        print("\n>>> [COMPLETE] Benchmark finished. Shutting down Colab runtime to save compute units...")
        time.sleep(5)  # Short buffer to ensure disk flush
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as e:
            print(f"\n>>> [WARN] Unable to unassign Colab runtime cleanly: {e}")


run_exp1(limit_samples=None, max_workers=1)

>>> Staging full_dev dataset from /content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627 to local NVMe (/content/sqlguard_run/full_dev)...
>>> Staging complete!

=================== RUNNING ABLATION EXPERIMENT: EXP-1_No_Repair (1534 SAMPLES) ===================


EXP-1_No_Repair:   0%|          | 0/1534 [00:00<?, ?it/s]

>>> Loading local SQL generator: seeklhy/codes-7b-bird-with-evidence...


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

...


[LIVE MONITOR] Evaluated: 1530/1534 (99.7%) | EX Acc: 48.89% | AST Valid: 96.9% | Repairs Recovered: 0/0 (0.0%)


EXP-1_No_Repair: 100%|█████████▉| 1532/1534 [3:03:55<00:13,  6.78s/it]


[LIVE MONITOR] Evaluated: 1532/1534 (99.9%) | EX Acc: 48.83% | AST Valid: 96.9% | Repairs Recovered: 0/0 (0.0%)


EXP-1_No_Repair: 100%|██████████| 1534/1534 [3:04:10<00:00,  7.20s/it]


{
  "experiment": "EXP-1_No_Repair",
  "total_samples": 1534,
  "ast_valid_pct": 96.870925684485,
  "samples_repaired": 0,
  "ex_count": 749,
  "ex_pct": 48.82659713168188
}

>>> [SYNC SUCCESS] Checkpointed ablation results to Drive: /content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627/ablation_results_separate/EXP-1_No_Repair/

>>> [COMPLETE] Benchmark finished. Shutting down Colab runtime to save compute units...


In [ ]:
pwd

'/content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627'

In [ ]:
ls

ablation_results_separate/  dev.sql               results/
dev_databases/              dev_tables.json       results_7b/
dev.json                    dev_tied_append.json  results_7b_with_evidence/
